# Detección automática de las áreas de análisis: métodos, estado y prueba de concepto

**Para qué sirve este cuaderno.** Explica, paso a paso y con figuras, cómo encuentra `interflib` las áreas de la
imagen en las que se lee la fracción de franja $F$, sin que haya que darle coordenadas. Cada método se presenta con
su idea, su ecuación, los parámetros que usa el código y una guía para leer cada figura. Al final se mide qué tan bien
funciona en las 22 series de análisis de la campaña 2026-10, dónde fallaba y cómo se corrigió.

**Actualización del 2026-10-28.** La primera versión de este cuaderno encontró bordes equivocados sin aviso con 5 a 20
franjas. Desde entonces la línea oscura del chaflán **confirma** el borde (secciones 3.6 y 3.7). Las secciones 5 y 6
muestran el balance de antes y el de ahora.

**Actualización del 2026-10-29.** ISO 3650 define la longitud central en el **punto central de la cara de medida
libre**. Por eso $F$ se lee ahora en el centro de la cara y no en la fila fija $y_e$. `locate_block` encuentra también
el resto del contorno de la cara, es decir, el borde lejano y los extremos (sección 3.9). `locate_face_centre` sitúa
el centro de tres maneras: con el contorno completo, con parte del contorno y las dimensiones de la cara, o con
coordenadas dadas a mano (sección 3.10). La sección 5 lo aplica a toda la campaña y muestra, en una galería, lo
detectado sobre cada uno de sus interferogramas.

**El problema.** Para leer $F$ en la columna del borde (comparando la fase del bloque con la de la platina en una
misma columna), los lectores de `interflib.fringe` necesitan:

- la **columna del borde** $x_e$, la frontera entre platina y bloque;
- la **fila de evaluación** $y_e$, donde se lee $F$. Desde el 2026-10-29 es la fila del **centro de la cara**,
  $(x_c, y_c)$, y las rectas de cada lado se prolongan hasta $x_c$;
- las **bandas** de columnas a cada lado del borde, de 30 a 180 px, de las que se toma la fase;
- la **ventana de ajuste**, $y_e \pm 75$ filas;
- las **filas de Fourier**, $y_e \pm 450$, que deben quedar sobre la cara del bloque.

Las bandas y las ventanas salen de $x_e$ y $y_e$, **siempre que** la imagen tenga el borde vertical y la platina a la
izquierda. Lo difícil es, por tanto, encontrar el borde y dejar la imagen en esa orientación.

**Antes**, $x_e$ se pasaba a mano (`--edge-x`) o se buscaba con una heurística que supone la geometría del banco
(sección 2). **Ahora**, `interflib.fringe.geometry.locate_block` lo encuentra con la demodulación de Fourier
(sección 3).

**Contenido.**

1. Las áreas que usan los lectores.
2. La heurística del banco y sus límites.
3. `locate_block`, método por método, sobre una imagen real:
   3.1 demodulación de Fourier; 3.2 zona con franjas; 3.3 mapa de saltos de fase; 3.4 picos por bandas de filas;
   3.5 RANSAC; 3.6 elección del borde; 3.7 la línea oscura: confirmación y afinado; 3.8 filas de la cara;
   3.9 contorno de la cara: borde lejano y extremos; 3.10 centro de la cara; 3.11 áreas resultantes.
4. Bloque girado y platina a la derecha: giro y espejo, en una escena simulada.
5. Balance en la campaña 2026-10.
6. Por qué fallaba y cómo se corrigió.
7. Estado y lo que falta.

**Glosario.**

| Término | Significado |
|---|---|
| Portadora $(g_x, g_y)$ | Frecuencia espacial de las franjas, en franjas por píxel. La da la inclinación del espejo de referencia. El periodo es $\Lambda = 1/g_y$. |
| Fondo $a$ y modulación $b$ | En $I = a + b\cos\varphi$: $a$ es la intensidad media y $b$ la amplitud de las franjas. Donde no hay franjas, $b \approx 0$. |
| Fase envuelta | Fase reducida a $[0, 2\pi)$: se conoce salvo múltiplos de $2\pi$. |
| Fase sin portadora $\psi$ | La fase menos la rampa de la portadora: lo que añade la superficie (bloque o platina) a la inclinación de la referencia. |
| Máscara | Imagen booleana que dice qué píxeles se usan. |
| Erosión | Operación sobre una máscara que le quita una capa de píxeles por todo su contorno. |
| Prominencia | Cuánto sobresale un pico de su entorno: su altura sobre el más alto de los dos valles que lo separan de picos más altos. |
| Inlier / outlier | Punto que sí / no se ajusta a un modelo, dentro de una tolerancia. |
| Subpíxel | Posición estimada con resolución menor que un píxel, interpolando una parábola por tres muestras. |
| Cara de medida | La cara libre del bloque (la de arriba en la imagen), de ancho $a_f$ y largo $b_f$ en mm. Su contorno son el borde junto a la platina $x_e$, el borde lejano $x_f$ y los extremos $y_t$ (arriba) y $y_b$ (abajo). |
| Centro de la cara $(x_c, y_c)$ | Punto central de la cara de medida, donde ISO 3650 define la longitud central. |
| Escala $m_{px}$ | Píxeles por milímetro sobre el objeto. |

Las secciones 2, 3, 5 y 6 usan las imágenes de la campaña (`base/data/raw`). Sin ellas, la sección 3 usa la escena
simulada (las figuras cambian) y las demás partes con datos reales se saltan.

## 0. Preparación

Se importan las funciones públicas de la librería y algunas internas de `interflib.fringe.geometry` (las que empiezan
por `_`). Usar las internas permite mostrar cada paso con el mismo código que ejecuta `locate_block`, sin copias que
puedan desincronizarse. `run_calibration` es un script de `tools/` y se carga como módulo.

In [ ]:
import importlib.util
import os
import tempfile
import time
from pathlib import Path

os.environ.setdefault("OPENCV_LOG_LEVEL", "ERROR")   # the camera TIFFs carry private tags

import matplotlib.pyplot as plt
import numpy as np
from IPython import get_ipython
from IPython.display import Image, display
from scipy.ndimage import binary_erosion, uniform_filter1d
from scipy.signal import find_peaks

from interflib.fringe import geometry as geo_mod
from interflib.fringe.edge_comparison import detect_edge_column, edge_line_centre
from interflib.fringe.fourier import estimate_carrier_2d, fourier_edge_fraction
from interflib.fringe.geometry import fringe_zone, locate_block, locate_face_centre, phase_step_map
from interflib.io.images import load_image
from interflib.simulation import PhaseModel, SceneGeometry, SensorModel, synthesize_interferogram

plt.rcParams.update({"figure.dpi": 100, "image.cmap": "gray", "font.size": 9})

# repository root: the first folder up from here that holds pyproject.toml (also from a copy in docs/)
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
TOOL = ROOT / "tools" / "run_calibration.py"
spec = importlib.util.spec_from_file_location("run_calibration", TOOL)
run_calibration = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run_calibration)

DATA = ROOT.parents[1] / "data" / "raw" / "202610_estudio-fraccion"
HAVE_DATA = DATA.exists()
REPET = DATA / "repetibilidad" / "20261005_L1_tg-inm_2mm_repet-2_serie-010"
print("campaign data:", DATA if HAVE_DATA else "not found")
OUT = Path(tempfile.mkdtemp())   # figures written by run_calibration --plot


def show_areas(argv, name):
    # Run the measurement and show the figure of the areas it used (the --plot figure).
    artifacts = {}
    summary = run_calibration.run(run_calibration.parse_args(argv), artifacts)
    path = OUT / f"{name}.png"
    run_calibration.plot_regions(path, summary, artifacts)   # it switches matplotlib to Agg
    get_ipython().run_line_magic("matplotlib", "inline")     # so the next figures show again
    display(Image(filename=str(path)))
    return summary


def load_series(folder, frames=5):
    # Mean of the first frames of a burst: averaging lowers the noise before the analysis.
    files = sorted(folder.glob("*.tif"))[:frames]
    return np.mean([load_image(f).astype(float) for f in files], axis=0)


def scene(rotation_deg=0.0, centre_mm=(0.0, 0.0), seed=0, size_mm=(9.0, 35.0)):
    # Simulated 2 mm + 51 nm block on its platen, CS505MU sensor at half scale (1024 x 1224 px).
    # The default 9 x 35 mm face is longer than the field: its ends are not in the image.
    geometry = SceneGeometry.cs505mu(scale=0.5, fov_mm=25.0, block_rotation_deg=rotation_deg,
                                     block_center_mm=centre_mm, block_width_mm=size_mm[0],
                                     block_height_mm=size_mm[1])
    model = PhaseModel(wavelength_nm=543.5, central_length_nm=2e6 + 51, fringes_y=30.0, fringes_x=1.0)
    return synthesize_interferogram(geometry, model, sensor=SensorModel.cs505mu(), seed=seed)


def bench_heuristic(img, rows):
    # The edge as located before locate_block (section 2).
    search, _ = run_calibration._search_from_platen(img)
    return edge_line_centre(img, detect_edge_column(img, rows=rows, search=search), rows=rows), search

## 1. Las áreas que usan los lectores

Se corre `run_calibration` sobre su escena simulada por defecto: un bloque de 2 mm + 51 nm con el borde vertical, en
el que la verdad es conocida. La figura es la misma que produce `--plot`.

**Cómo leer la figura.**

- **(a) Cuadro normalizado.** La imagen completa, ya con el borde vertical y la platina a la izquierda. La línea
  naranja es el borde $x_e$ encontrado, la franja verde son las filas de Fourier, las líneas rojas punteadas las filas
  en que la cara tiene franjas, y el recuadro blanco la zona ampliada en (b).
- **(b) Áreas alrededor del borde.** El recuadro azul agrupa las bandas de la platina (a la izquierda) y el rojo las
  del bloque (a la derecha), en la altura de la ventana de ajuste. Las líneas discontinuas marcan las regiones de la
  variante 2-D de Fourier. La franja verde clara son las columnas de las que se estima el periodo, y la línea punteada
  blanca la fila de evaluación.
- **(c) Fases de las bandas.** Cada punto es la fase de una banda en la fila de evaluación, en franjas, relativa a la
  platina en el borde. Las rectas son el ajuste de cada lado prolongado hasta el borde (línea naranja). **$F$ es el
  salto entre las dos rectas en el borde.**

In [ ]:
sim = show_areas(["--scale", "0.5", "--frames", "2"], "simulated")
print(f"edge x = {sim['image']['edge_x_px']:.2f} px (from the simulated mask: {sim['truth']['edge_x_px']:.2f} px)")
print(f"F = {sim['fraction']['fraction_used']:.4f}, truth {sim['truth']['fraction']:.4f}; "
      f"length error {sim['truth']['length_error_nm']:+.2f} nm")

El borde se encontró a menos de medio píxel del verdadero y $F$ coincide con la verdad a $10^{-4}$ franjas. No se dio
ninguna coordenada a mano.

## 2. La heurística del banco y sus límites

Antes de `locate_block`, el borde se buscaba con tres pasos sobre la intensidad, pensados para el banco del INM. Se
calculan en las filas 600–1600:

1. **Inicio de la franja de platina.** Para cada columna $x$ se calcula el contraste de franjas, la desviación típica
   de la intensidad a lo largo de las filas, $c(x) = \operatorname{std}_y I(x, y)$, suavizada en 9 columnas. La franja
   de platina empieza en la primera columna con $c(x) > 0{,}4\,c_{95}$, donde $c_{95}$ es el percentil 95 de $c$.
2. **Salto de intensidad.** En el tramo de 150 a 400 px a la derecha de ese inicio, donde en el banco está siempre el
   borde, se busca el máximo del salto medio entre columnas vecinas,
   $J(x) = \langle |I(x+1, y) - I(x, y)| \rangle_y$, suavizado en 5 columnas. En el borde las franjas de bloque y
   platina están desfasadas y el salto es grande.
3. **Línea oscura.** El chaflán del bloque se ve como una línea oscura de unos 10 px. A ±15 px del salto se toma la
   columna de menor intensidad media y se afina con una parábola por tres puntos (posición subpíxel).

**Cómo leer la figura** (imagen real del 2 mm, `repet-2`).

- **Arriba:** el contraste por columna, $c(x)$, con su umbral (rojo discontinuo), el inicio de la platina (azul) y el
  tramo de búsqueda (naranja claro).
- **En medio:** el salto $J(x)$ dentro del tramo, y su máximo.
- **Abajo:** la intensidad media por columna cerca del salto. La zona gris es la ventana de ±15 px; la línea naranja,
  el mínimo de la línea oscura, que es el borde final.

Qué hay que notar:

- En el contraste se ven la franja de platina (desde x ≈ 849), una caída brusca en el borde del bloque (la línea
  oscura apaga las franjas) y el final del campo con franjas hacia x ≈ 1650.
- El salto $J(x)$ tiene **dos picos**, uno a cada flanco de la línea oscura, porque es ahí donde la intensidad cambia
  de golpe. Por eso el paso 3 no se queda con el máximo de $J$, sino con el mínimo de la línea oscura, que está entre
  los dos.

In [ ]:
if HAVE_DATA:
    img_real = load_series(REPET, frames=30)
    y0, y1 = run_calibration.BENCH_EDGE_ROWS
    contrast = np.convolve(img_real[y0:y1].std(axis=0), np.ones(9) / 9, "same")
    threshold = 0.4 * np.percentile(contrast, 95)
    left = int(np.flatnonzero(contrast > threshold).min())
    search = (left + 150, left + 400)
    jump = np.convolve(np.abs(np.diff(img_real[y0:y1, search[0]:search[1]], axis=1)).mean(axis=0),
                       np.ones(5) / 5, "same")
    step_col = detect_edge_column(img_real, rows=(y0, y1), search=search)
    x_heur = edge_line_centre(img_real, step_col, rows=(y0, y1))
    means = np.convolve(img_real[y0:y1].mean(axis=0), np.ones(3) / 3, "same")

    fig, ax = plt.subplots(3, 1, figsize=(9, 7))
    ax[0].plot(contrast, color="k", lw=1)
    ax[0].axhline(threshold, color="tab:red", ls="--", lw=1, label="umbral 0,4·c95")
    ax[0].axvline(left, color="tab:blue", lw=1, label=f"inicio de la platina, x = {left}")
    ax[0].axvspan(*search, color="tab:orange", alpha=0.15, label="tramo de búsqueda (+150 a +400 px)")
    ax[0].set_ylabel("contraste c(x)")
    ax[0].legend(fontsize=7, loc="upper right")
    xs = np.arange(search[0], search[1] - 1) + 1
    ax[1].plot(xs, jump, color="k", lw=1)
    ax[1].axvline(step_col, color="tab:orange", lw=1, label=f"máximo del salto, x = {step_col}")
    ax[1].set_ylabel("salto J(x)")
    ax[1].legend(fontsize=7)
    xz = np.arange(step_col - 40, step_col + 41)
    ax[2].plot(xz, means[xz], color="k", lw=1)
    ax[2].axvspan(step_col - 15, step_col + 15, color="0.9", label="±15 px alrededor del salto")
    ax[2].axvline(x_heur, color="tab:orange", lw=1.5, label=f"línea oscura, x = {x_heur:.2f}")
    ax[2].set_xlabel("columna x (px)")
    ax[2].set_ylabel("intensidad media")
    ax[2].legend(fontsize=7)
    fig.tight_layout()
else:
    print("no campaign data: figure skipped")

**Sus límites.** La heurística funciona en el banco porque allí se cumplen sus tres supuestos: la platina está a la
izquierda, la franja de platina mide unos 240 px (así que el borde cae en el tramo de búsqueda) y la línea oscura se
ve. No sabe qué hacer con un bloque girado, con la platina a la derecha o con otro aumento.

En la escena simulada, que no dibuja línea oscura y cuya platina ocupa todo el campo, la heurística toma un mínimo de
intensidad cualquiera, y con el bloque girado y desplazado busca además en el tramo equivocado:

In [ ]:
straight, tilted = scene(), scene(rotation_deg=8.0, centre_mm=(2.0, 0.0))
for label, s in (("straight block", straight), ("block rotated 8 deg, shifted 2 mm", tilted)):
    im_s = s.image.astype(float)
    y_mid = im_s.shape[0] // 2
    true_edge = np.flatnonzero(s.block_mask[y_mid])[0] - 0.5
    x_h, rng_h = bench_heuristic(im_s, (0, im_s.shape[0]))
    print(f"{label:34s} true edge {true_edge:6.1f} px | heuristic searches {rng_h[0]}-{rng_h[1]}, "
          f"finds {x_h:6.1f} px, off by {x_h - true_edge:+6.1f} px")

## 3. `locate_block`, método por método

La idea general es no depender de la intensidad, que cambia con la iluminación, sino de la **fase** de las franjas.
Bloque y platina son dos superficies a distinta altura. Sus franjas tienen la misma portadora, pero están desfasadas
$2\pi F$ (más una pequeña cuña). Si se obtiene la fase de cada píxel y se le quita la portadora, el bloque aparece como
una región con otra fase, y su contorno como una línea de saltos de fase, con cualquier orientación.

Se usa la imagen real del 2 mm (`repet-2`, media de 30 cuadros), o la escena simulada girada si no hay datos.

In [ ]:
img = img_real if HAVE_DATA else tilted.image.astype(float)
height, width = img.shape
y_e = run_calibration.BENCH_EVAL_ROW if HAVE_DATA else height // 2
print(f"image {height} x {width} px ({'2 mm, repet-2' if HAVE_DATA else 'simulated, rotated 8 deg'}); "
      f"evaluation row {y_e}")

### 3.1 Demodulación de Fourier: de la intensidad a la fase

**Modelo.** Con el espejo de referencia inclinado, la intensidad de un interferograma de dos haces es

$$I(x,y) = a(x,y) + b(x,y)\cos\bigl[2\pi(g_x x + g_y y) + \psi(x,y)\bigr],$$

con $a$ el fondo, $b$ la modulación, $(g_x, g_y)$ la portadora y $\psi$ la fase que añade la superficie. Escribiendo
el coseno como suma de dos exponenciales,

$$I = a + \tfrac{b}{2}\,e^{i\psi}\,e^{\,i2\pi(g_x x + g_y y)} + \tfrac{b}{2}\,e^{-i\psi}\,e^{-i2\pi(g_x x + g_y y)}.$$

Por eso el espectro de Fourier de la imagen tiene **tres lóbulos**: el del fondo $a$, cerca de la frecuencia cero, y
los dos de las franjas, centrados en $+(g_x, g_y)$ y en $-(g_x, g_y)$ [Takeda, Ina & Kobayashi, *J. Opt. Soc. Am.* 72,
156–160 (1982)]. Si $a$, $b$ y $\psi$ varían despacio frente al periodo de las franjas, los lóbulos son estrechos y no
se tocan.

**Cómo se recupera la fase: detección síncrona** [Servín, Quiroga & Padilla, *Fringe Pattern Analysis for Optical
Metrology*, Wiley-VCH (2014), §4.2].

1. Se multiplica la imagen por $e^{-i2\pi(g_x x + g_y y)}$. El lóbulo de $+g$ se traslada a la frecuencia cero, y el
   fondo y el otro lóbulo se alejan a $-g$ y $-2g$.
2. Se aplica un **filtro paso bajo** circular de radio $0{,}5\,|g|$, con borde de coseno alzado (cae suave a cero, para
   no crear ondulaciones). Deja pasar solo el lóbulo trasladado, $\frac{b}{2}\,e^{i\psi}$.
3. De ese número complejo por píxel se leen la **modulación** $b$ (dos veces su módulo) y la **fase sin portadora**
   $\psi$ (su ángulo), envuelta en $[0, 2\pi)$.

**Cómo se estima la portadora.** Se toma la mitad central de las filas y las columnas de más contraste, se les resta la
media y se multiplican por una ventana de Hann, que lleva el recorte a cero en sus bordes para que el corte no cree
frecuencias falsas. Se calcula el espectro de potencia con relleno de ceros ×4 (que muestrea el espectro más fino) y la
portadora es el pico más alto fuera de la frecuencia cero, en el semiplano $g_y > 0$ (los dos lóbulos son simétricos),
afinado con una parábola por el logaritmo de los tres puntos vecinos. **Si el pico da menos de 4 franjas en la altura
del recorte, se declara que no hay franjas y la función falla.**

**Cómo leer la figura.**

- **Izquierda:** el logaritmo del espectro de potencia del recorte, centrado en la frecuencia cero y ampliado a las
  frecuencias bajas. El lóbulo central es el fondo; los de arriba y abajo, las franjas (que son casi horizontales, así
  que la portadora apunta casi a lo largo de $f_y$). La cruz cian marca la portadora estimada, y el círculo cian el
  filtro de radio $0{,}5|g|$ alrededor de ella: lo que queda dentro es lo que se usa. Las estelas horizontales que
  salen de cada lóbulo vienen de que las franjas terminan de golpe a los lados (bordes de la zona y del bloque): un
  corte brusco en $x$ reparte energía a lo largo de $f_x$. Las manchas pequeñas sueltas son otras estructuras débiles
  de la imagen.
- **Derecha:** la intensidad de una columna de la platina a lo largo de las filas. Es casi un coseno, de periodo
  $1/g_y$. Su amplitud $b$ y su valor medio $a$ crecen hacia abajo porque el haz es más intenso en el centro: la
  demodulación tolera que $a$ y $b$ cambien así, siempre que lo hagan despacio frente al periodo.

**Una consecuencia del filtro.** Un filtro de radio $0{,}5|g|$ en frecuencia equivale a promediar la fase en unos
$1/(0{,}5|g|)$ píxeles: aquí unos 73 px. Todo lo que cambia más rápido, como el salto de fase en el borde del bloque,
sale **suavizado** en ese ancho. Esto importa en 3.3.

In [ ]:
rows_c = slice(height // 4, 3 * height // 4)
contrast_c = uniform_filter1d(img[rows_c].std(axis=0), 9)
cols_c = np.flatnonzero(contrast_c > 0.4 * np.percentile(contrast_c, 95))
region = img[rows_c, cols_c.min():cols_c.max() + 1]
g_x, g_y = estimate_carrier_2d(region)
print(f"carrier g_x = {g_x:+.5f}, g_y = {g_y:.5f} fringe/px: period {1 / g_y:.1f} px, "
      f"{g_y * region.shape[0]:.0f} fringes over the {region.shape[0]} rows of the crop, "
      f"fringes tilted {np.degrees(np.arctan2(g_x, g_y)):+.2f} deg from the rows")

ny, nx = region.shape
centred = (region - region.mean()) * np.outer(np.hanning(ny), np.hanning(nx))
power = np.fft.fftshift(np.abs(np.fft.fft2(centred)) ** 2)
fy = np.fft.fftshift(np.fft.fftfreq(ny))
fx = np.fft.fftshift(np.fft.fftfreq(nx))
lim = 2.0 * g_y

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 1.3]})
ax[0].imshow(np.log10(power + 1), extent=(fx[0], fx[-1], fy[-1], fy[0]), cmap="inferno", aspect="auto")
ax[0].plot(g_x, g_y, "c+", ms=12, mew=2)
ax[0].add_patch(plt.Circle((g_x, g_y), 0.5 * np.hypot(g_x, g_y), fill=False, ec="c", lw=1.2))
ax[0].set_xlim(-lim, lim)
ax[0].set_ylim(lim, -lim)
ax[0].set_xlabel("$f_x$ (franjas/px)")
ax[0].set_ylabel("$f_y$ (franjas/px)")
ax[0].set_title("espectro de potencia (log), portadora y filtro")
col = cols_c.min() + 20
ax[1].plot(np.arange(height), img[:, col], color="k", lw=0.6)
ax[1].set_xlim(y_e - 200, y_e + 200)
ax[1].set_xlabel("fila y (px)")
ax[1].set_ylabel("intensidad (cuentas)")
ax[1].set_title(f"una columna de la platina (x = {col}): periodo {1 / g_y:.1f} px")
fig.tight_layout()

### 3.2 La zona con franjas

**Qué es.** Los píxeles donde hay franjas de verdad: la parte iluminada de bloque y platina. Fuera de ella la fase no
significa nada, y cualquier salto de fase sería ruido.

**Cómo se obtiene.** Se demodula el campo completo (3.1) y se umbraliza la modulación:

$$\text{zona} = \{\,b \ge 0{,}3\,b_{95}\,\},$$

con $b_{95}$ el percentil 95 de $b$. Se usa la modulación y no la intensidad porque una zona brillante sin franjas tiene
intensidad alta pero $b \approx 0$. Se usa el percentil 95 y no la mediana porque en el banco más de la mitad del
cuadro es fondo oscuro: la mediana de $b$ sería casi cero y el umbral no separaría nada.

**Cómo leer la figura.**

- **Izquierda:** la imagen con el contorno de la zona en verde. El contorno pequeño de dentro, sobre el borde del bloque
  hacia la fila 1300, es un **hueco** de la zona: allí $b$ cae por debajo del umbral (se ve como una mancha oscura en
  el mapa del centro).
- **Centro:** el mapa de modulación $b$, claro donde las franjas tienen mucha amplitud. La línea vertical oscura que lo
  cruza es el borde del bloque, donde las franjas pierden contraste por el chaflán. Las variaciones suaves de brillo
  son la forma del haz.
- **Derecha:** el histograma de $b$ (eje vertical logarítmico) con el umbral en rojo. El pico de la izquierda es el
  fondo oscuro; la cola de la derecha, la zona con franjas.

In [ ]:
phase_map, zone = fringe_zone(img)
b_map = phase_map.modulation
b95 = np.nanpercentile(b_map, 95)
print(f"b95 = {b95:.1f} counts, threshold {0.3 * b95:.1f}; the zone covers {100 * zone.mean():.0f} % of the frame")

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 1.15, 0.9]})
ax[0].imshow(img)
ax[0].contour(zone, [0.5], colors="lime", linewidths=1)
ax[0].set_title("imagen y contorno de la zona")
im = ax[1].imshow(b_map, cmap="magma")
fig.colorbar(im, ax=ax[1], shrink=0.8, label="b (cuentas)")
ax[1].set_title("modulación b")
for a in ax[:2]:
    a.set_xlabel("columna x (px)")
    a.set_ylabel("fila y (px)")
ax[2].hist(b_map[np.isfinite(b_map)].ravel(), bins=120, color="0.4", log=True)
ax[2].axvline(0.3 * b95, color="tab:red", ls="--", label="umbral 0,3·b95")
ax[2].axvline(b95, color="tab:blue", ls=":", label="b95")
ax[2].set_xlabel("b (cuentas)")
ax[2].set_ylabel("píxeles (escala log)")
ax[2].legend(fontsize=7)
ax[2].set_title("histograma de la modulación")
fig.tight_layout()

### 3.3 El mapa de saltos de fase

**Qué se busca.** Al quitar la portadora, la fase $\psi$ es casi constante sobre la platina y casi constante sobre el
bloque, pero las dos difieren en $2\pi F$ más la cuña. El contorno del bloque es, por tanto, una línea donde $\psi$
salta.

**Cómo se mide el salto.** La fase está envuelta: un salto de $0{,}1$ rad y uno de $2\pi - 0{,}1$ rad son el mismo. Por
eso no se restan fases, sino fasores $z = e^{i\psi}$: la diferencia envuelta entre dos píxeles es el ángulo de
$z_1\,\overline{z_2}$, siempre en $(-\pi, \pi]$. El mapa de saltos combina las diferencias centradas en $x$ y en $y$,

$$S(x,y) = \sqrt{\Bigl|\arg\bigl(z_{x+1,y}\,\overline{z_{x-1,y}}\bigr)\Bigr|^2 + \Bigl|\arg\bigl(z_{x,y+1}\,\overline{z_{x,y-1}}\bigr)\Bigr|^2},$$

promediado en una ventana de 5 × 5 px para bajar el ruido. $S$ es grande donde la fase cambia de golpe y pequeño donde
varía suave. Como usa las dos direcciones, marca bordes con cualquier orientación.

**El salto es pequeño.** Por el suavizado del filtro (3.1), un salto de $2\pi F$ se reparte en unos 70 px. Con
$F = 0{,}18$ el salto total es de unos 1,1 rad, así que la diferencia entre columnas vecinas es de solo unas centésimas
de radián. El borde no es una línea muy brillante, sino un relieve débil pero sistemático sobre un fondo casi plano.
Fuera de la zona, en cambio, la fase es ruido y $S$ es grande: por eso todo lo que sigue mira solo dentro de la zona.

**Una limitación.** Si $F \approx 0$ (o $\approx 1$), bloque y platina tienen casi la misma fase y el borde casi no se
ve en $S$. En ese caso solo lo marca la cuña. En esta imagen pasa de verdad, como se ve en 3.5.

**Cómo leer la figura.**

- **Izquierda:** la fase sin portadora $\psi$ con una paleta cíclica (el color de $0$ y el de $2\pi$ son el mismo), y el
  contorno de la zona. Dentro de la zona, la platina y el bloque son regiones de color casi uniforme y distinto entre
  sí. Fuera, el color es ruido: allí no hay franjas.
- **Centro:** el mapa de saltos $S$ **solo dentro de la zona** (fuera, en gris), con la escala de color ajustada a los
  valores de dentro. El borde del bloque es la línea vertical clara hacia x ≈ 1088, más marcada en la mitad de abajo;
  el borde derecho de la zona también se marca, porque allí la fase deja de tener sentido. La línea punteada blanca es
  la fila del perfil de la derecha.
- **Derecha:** $S$ a lo largo de esa fila, dentro de la zona. El pico central (unos 0,03 rad) es el borde del bloque;
  los de los extremos, las orillas de la zona.

In [ ]:
step = phase_step_map(phase_map)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 1.15, 1.1]})
ax[0].imshow(phase_map.object_phase, cmap="twilight")
ax[0].contour(zone, [0.5], colors="lime", linewidths=0.6)
ax[0].set_title("fase sin portadora ψ (paleta cíclica)")
cmap_s = plt.get_cmap("viridis").copy()
cmap_s.set_bad("0.3")
im = ax[1].imshow(np.ma.masked_where(~zone, step), cmap=cmap_s, vmin=0, vmax=np.percentile(step[zone], 99))
fig.colorbar(im, ax=ax[1], shrink=0.8, label="S (rad)")
ax[1].axhline(y_e, color="w", ls=":", lw=0.8)
ax[1].set_title("mapa de saltos S dentro de la zona")
for a in ax[:2]:
    a.set_xlabel("columna x (px)")
    a.set_ylabel("fila y (px)")
in_row = np.flatnonzero(zone[y_e])
ax[2].plot(in_row, step[y_e, in_row], color="k", lw=0.8)
ax[2].set_xlabel("columna x (px)")
ax[2].set_ylabel("S (rad)")
ax[2].set_title(f"perfil de S en la fila {y_e}, dentro de la zona")
fig.tight_layout()

### 3.4 Picos por bandas de filas

**Por qué por bandas.** Una sola fila es ruidosa, y promediar todas las filas emborronaría un borde inclinado en una
mancha ancha. Se corta el campo en **bandas de 40 filas**: dentro de cada una, el borde es casi un punto, y la sucesión
de puntos de banda en banda dibuja la línea, recta o inclinada.

**Qué se hace en cada banda.**

1. **Solo dentro de la zona, y lejos de su orilla.** La zona se erosiona 5 px y se promedia $S$ por columna solo sobre
   los píxeles que quedan.
2. **Picos con prominencia.** Se buscan los picos del perfil con `scipy.signal.find_peaks` y se exige una prominencia
   de al menos el 30 % del máximo del perfil. La prominencia mide cuánto sobresale el pico sobre los valles que lo
   rodean, así que descarta las ondulaciones del ruido aunque estén altas.
3. **Lejos del borde de la zona.** Se descartan los picos a menos de 40 px de la primera o la última columna cubierta.
   Así la orilla del haz, que también da un salto de fase, no cuenta como borde del bloque.
4. **Posición subpíxel.** Cada pico se afina con la parábola que pasa por él y por sus dos vecinos.

**Cómo leer la figura.** Es el perfil de la banda de 40 filas centrada en la fila de evaluación. La línea negra es el
promedio de $S$ por columna dentro de la zona erosionada. Las zonas grises son los 40 px excluidos junto a cada orilla
de la zona. Los triángulos rojos son los picos aceptados y los grises los descartados por estar en la orilla; la barra
vertical bajo cada triángulo es su prominencia. En esta banda el borde da un pico de unos 0,03 rad en x ≈ 1089; los
picos de las orillas son tan altos o más, y por eso hace falta excluirlos.

In [ ]:
s0 = y_e - geo_mod.SLAB_ROWS // 2
s1 = s0 + geo_mod.SLAB_ROWS
inner = binary_erosion(zone[s0:s1], iterations=5)
counts = inner.sum(axis=0)
profile = np.where(inner, step[s0:s1], 0.0).sum(axis=0) / np.maximum(counts, 1)
covered = np.flatnonzero(counts > 0)
lo, hi = covered.min() + geo_mod.BORDER_MARGIN, covered.max() - geo_mod.BORDER_MARGIN
peaks, props = find_peaks(profile, prominence=0.3 * profile.max())
accepted = geo_mod._interior_peaks(step, zone, s0, s1)
print(f"band rows {s0}-{s1}: {len(peaks)} peaks with prominence >= 30 % of the maximum; "
      f"accepted (away from the zone border): {[round(float(p), 1) for p, _ in accepted]}")

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(profile, color="k", lw=0.8)
ax.axvspan(covered.min(), lo, color="0.85")
ax.axvspan(hi, covered.max(), color="0.85", label="40 px junto a la orilla (excluidos)")
for p, prom in zip(peaks, props["prominences"]):
    color = "tab:red" if lo <= p <= hi else "0.5"
    ax.plot(p, profile[p], "v", color=color, ms=8)
    ax.vlines(p, profile[p] - prom, profile[p], color=color, lw=1)
ax.plot([], [], "v", color="tab:red", label="pico aceptado (barra: prominencia)")
ax.plot([], [], "v", color="0.5", label="pico en la orilla, descartado")
ax.set_xlim(covered.min() - 50, covered.max() + 50)
ax.set_xlabel("columna x (px)")
ax.set_ylabel("S medio en la banda (rad)")
ax.set_title(f"perfil de saltos de una banda de {geo_mod.SLAB_ROWS} filas")
ax.legend(fontsize=7)
fig.tight_layout()

Repitiendo esto en todas las bandas se obtiene una nube de puntos $(y, x)$: un punto por pico y por banda.

In [ ]:
rows_with_fringes = np.flatnonzero(zone.any(axis=1))
y_lo, y_hi = rows_with_fringes.min(), rows_with_fringes.max()
starts = np.arange(y_lo, max(y_lo + 1, y_hi - geo_mod.SLAB_ROWS), geo_mod.SLAB_ROWS)
points = [((b0 + min(b0 + geo_mod.SLAB_ROWS, height)) / 2, x, prom) for b0 in starts
          for x, prom in geo_mod._interior_peaks(step, zone, b0, min(b0 + geo_mod.SLAB_ROWS, height))]
pts = np.array(points)
print(f"{len(starts)} bands of {geo_mod.SLAB_ROWS} rows between rows {y_lo} and {y_hi}; {len(points)} peaks in total")

### 3.5 RANSAC: encontrar líneas en una nube de puntos con intrusos

**El problema.** Los puntos del borde están alineados, pero la nube trae además **intrusos** (*outliers*): picos de
polvo, de franjas parásitas, de una raya o de otro borde. Un ajuste por mínimos cuadrados usa todos los puntos a la vez:
cada intruso tira de la recta hacia él, y basta uno lejano para torcerla. Además, mínimos cuadrados da siempre **una**
recta, y en la imagen puede haber dos bordes (un bloque rodeado de platina tiene dos).

**La idea de RANSAC** (*RANdom SAmple Consensus*, «consenso de muestras aleatorias») [Fischler & Bolles, *Commun.
ACM* 24, 381–395 (1981)]: en vez de ajustar con todos los puntos, se proponen muchas rectas candidatas con muy pocos
puntos y se elige la que **más puntos explica**. Una recta queda fijada por dos puntos, así que:

1. Se eligen **dos puntos al azar** y se traza la recta que pasa por ellos, $x = a + b\,y$. Se escribe $x$ en función
   de $y$ porque el borde es casi vertical: así la pendiente $b$ es pequeña y no diverge.
2. Se descarta si se aparta más de **30°** de la vertical.
3. Se cuentan sus **inliers**: los puntos a **3 px** o menos de la recta, medidos en $x$.
4. Se repite **300 veces** y se guarda la recta con más inliers: el «consenso».
5. Se **reajusta** esa recta por mínimos cuadrados, pero solo con sus inliers, que ya no traen intrusos.
6. Se **quitan** esos inliers de la nube y se vuelve al paso 1, para buscar otra línea.
7. Se para cuando la mejor recta tiene puntos en menos de **6 bandas** distintas (unas 240 filas): eso ya no es un
   borde, sino puntos sueltos.

**Por qué funciona.** Si la mitad de los puntos son del borde, la probabilidad de que dos puntos al azar sean del borde
es 1/4, y la de no acertar nunca en 300 intentos es $(3/4)^{300} \approx 10^{-38}$. La semilla del azar está fija
(`default_rng(0)`), así que el resultado es reproducible.

**Ejemplo de juguete.** Antes de usarlo en la imagen, se prueba con puntos inventados: dos bordes casi verticales en
$x \approx 300$ y $x \approx 700$ (pendiente 0,05, unos 3°), con ruido de 1 px, más 15 intrusos al azar.

**Cómo leer la figura.**

- **Izquierda:** la nube de puntos (negro); la recta de mínimos cuadrados con todos los puntos (gris discontinua), que
  no se parece a ningún borde; y **un solo intento** de RANSAC: los dos puntos elegidos al azar (estrellas rojas), la
  recta que definen (naranja) y su banda de ±3 px. Los puntos dentro de la banda son los inliers de ese intento.
- **Derecha:** las líneas que encuentra RANSAC tras los 300 intentos, cada una con sus inliers en su color, y los
  intrusos que quedan sin asignar (negro).

In [ ]:
rng = np.random.default_rng(3)
ys = np.arange(20, 1000, 40.0)
true_lines = [(300.0, 0.05), (700.0, 0.05)]
toy = [(y, a + b * y + rng.normal(0, 1.0), 1.0) for a, b in true_lines for y in ys]
toy += [(rng.uniform(0, 1000), rng.uniform(0, 1000), 1.0) for _ in range(15)]
toy = np.array(toy)

found = geo_mod._edge_lines(toy, len(ys), np.random.default_rng(0))
b_ls, a_ls = np.polyfit(toy[:, 0], toy[:, 1], 1)
yy = np.array([0.0, 1000.0])

fig, ax = plt.subplots(1, 2, figsize=(11, 4.6), sharey=True)
ax[0].plot(toy[:, 1], toy[:, 0], "k.", ms=5, label="puntos")
ax[0].plot(a_ls + b_ls * yy, yy, "--", color="0.5", lw=1.5, label="mínimos cuadrados (todos)")
i, j = 4, 10                                            # one RANSAC hypothesis, for the picture
(yi, xi), (yj, xj) = toy[i, :2], toy[j, :2]
bh = (xj - xi) / (yj - yi)
ah = xi - bh * yi
inl = np.abs(toy[:, 1] - (ah + bh * toy[:, 0])) <= 3.0
ax[0].fill_betweenx(yy, ah + bh * yy - 3, ah + bh * yy + 3, color="tab:orange", alpha=0.3)
ax[0].plot(ah + bh * yy, yy, color="tab:orange", lw=1, label=f"un intento: {inl.sum()} inliers")
ax[0].plot([xi, xj], [yi, yj], "*", color="tab:red", ms=12, label="los 2 puntos elegidos")
ax[0].set_title("mínimos cuadrados frente a un intento de RANSAC")
remaining = np.ones(len(toy), bool)
for k, (a, b, support) in enumerate(found):
    member = remaining & (np.abs(toy[:, 1] - (a + b * toy[:, 0])) <= 3.0)
    ax[1].plot(toy[member, 1], toy[member, 0], ".", ms=7, color=f"C{k}")
    ax[1].plot(a + b * yy, yy, color=f"C{k}", lw=1.5, label=f"línea {k + 1}: en {100 * support:.0f} % de las bandas")
    remaining &= ~member
ax[1].plot(toy[remaining, 1], toy[remaining, 0], "k.", ms=6, label="sin asignar (intrusos)")
ax[1].set_title("líneas encontradas por RANSAC")
for a in ax:
    a.set_xlabel("x (px)")
    a.set_xlim(0, 1000)
    a.legend(fontsize=7, loc="lower right")
ax[0].set_ylabel("y (px)")
ax[0].set_ylim(1000, 0)
fig.tight_layout()
print(f"least squares with all points: x = {a_ls:.1f} + {b_ls:.4f} y")
for k, (a, b, support) in enumerate(found):
    print(f"RANSAC line {k + 1}: x = {a:.1f} + {b:.4f} y")
print("true lines: x = 300 + 0.05 y and x = 700 + 0.05 y")

RANSAC recupera las dos rectas a pesar de los intrusos, mientras que mínimos cuadrados da una recta que no corresponde
a nada.

**En la imagen real** se hace lo mismo con los picos de las bandas (3.4). Cada línea que sale se lista con su posición
en la fila media, su ángulo respecto a la vertical y su **soporte**: la fracción de bandas en que aparece.

In [ ]:
lines = geo_mod._edge_lines(points, len(starts), np.random.default_rng(0))
y_ref = (y_lo + y_hi) / 2
best_support = max(line[2] for line in lines)
for a, b, support in lines:
    strong = support >= geo_mod.STRONG_FRACTION * best_support
    print(f"x = {a + b * y_ref:7.1f} px at row {y_ref:.0f}, {np.degrees(np.arctan(b)):+6.2f} deg, "
          f"in {100 * support:3.0f} % of the bands -> {'strong' if strong else 'weak, discarded'}")
print(f"{len(starts)} bands in all: the edge line is found in {round(best_support * len(starts))} of them")

**Por qué el borde aparece solo en una parte de las bandas.** La cuña entre bloque y platina hace que $F$ cambie a lo
largo de $y$, y con ella el tamaño del salto de fase.

**Cómo leer la figura.**

- **Izquierda:** para cada banda de 40 filas, el valor de $S$ en el borde (el máximo del perfil de la banda a ±12 px
  del borde), en negro, y el máximo de todo el perfil de la banda, en gris. Los puntos rojos son las bandas en las que
  el borde se aceptó como pico.
- **Derecha:** $F$ leída en el borde con la ventana de ajuste centrada en distintas filas.

En las filas de arriba $F$ se acerca a 0 (0,03 en la fila 720): bloque y platina tienen casi la misma fase, no hay
salto y el borde no se encuentra. Es la limitación de 3.3, en datos reales. Hacia la fila 1300 el pico se parte en dos,
a cada lado del hueco de la zona (3.2), y esos puntos quedan fuera de la recta. Abajo dominan los picos de la orilla.
El borde se encuentra en las bandas centrales, y eso basta, porque RANSAC solo necesita 6.

In [ ]:
edge_a, edge_b, _ = max(lines, key=lambda line: line[2])   # the edge line found by RANSAC
band_rows, at_edge, band_max, hit = [], [], [], []
for b0 in starts:
    b1 = min(b0 + geo_mod.SLAB_ROWS, height)
    inner_b = binary_erosion(zone[b0:b1], iterations=5)
    cnt = inner_b.sum(axis=0)
    if cnt.sum() == 0:
        continue
    prof_b = np.where(inner_b, step[b0:b1], 0.0).sum(axis=0) / np.maximum(cnt, 1)
    xe_b = int(round(edge_a + edge_b * (b0 + b1) / 2))
    band_rows.append((b0 + b1) / 2)
    at_edge.append(prof_b[xe_b - 12:xe_b + 13].max())
    band_max.append(prof_b.max())
    hit.append(any(abs(x - xe_b) <= geo_mod.INLIER_PX for x, _ in geo_mod._interior_peaks(step, zone, b0, b1)))
band_rows, at_edge, band_max, hit = map(np.array, (band_rows, at_edge, band_max, hit))

geo_f = locate_block(img, eval_row=y_e)
lo_f, hi_f = geo_f.face_rows
f_rows = [y for y in range(lo_f + 80, hi_f - 80, 60)]
f_vals = [fourier_edge_fraction(geo_f.image, geo_f.edge_x, (y - 75, y + 75), y, fft_rows=(lo_f, hi_f)).fraction
          for y in f_rows]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(band_rows, band_max, "o-", color="0.7", ms=3, label="máximo del perfil de la banda")
ax[0].plot(band_rows, at_edge, "o-", color="k", ms=3, label="S en el borde (±12 px)")
ax[0].plot(band_rows[hit], at_edge[hit], "o", color="tab:red", ms=6, label="borde aceptado como pico")
ax[0].set_xlabel("fila central de la banda (px)")
ax[0].set_ylabel("S (rad)")
ax[0].set_title("fuerza del borde en cada banda")
ax[0].legend(fontsize=7)
ax[1].plot(f_rows, f_vals, "o-", color="tab:blue", ms=4)
ax[1].axhline(0, color="0.6", lw=0.8)
ax[1].set_ylim(-0.05, 1.0)
ax[1].set_xlabel("fila de evaluación (px)")
ax[1].set_ylabel("F en el borde")
ax[1].set_title("F a lo largo de y (por la cuña)")
fig.tight_layout()
print("F along y:", ", ".join(f"{y}: {f:.3f}" for y, f in zip(f_rows, f_vals)))

### 3.6 Cuál de las líneas es el borde

Tres criterios, en este orden:

1. **Líneas débiles.** Se descartan las líneas con menos del **70 %** del soporte de la mejor. Una raya, una franja
   parásita o un trozo de borde suelto aparecen en pocas bandas; el borde del bloque, en muchas.
2. **Línea oscura.** Entre las líneas fuertes solo compiten las que llevan a lo largo la línea oscura del chaflán
   (sección 3.7). Una línea de saltos de fase sin línea oscura puede ser un artefacto de la demodulación: con franjas
   anchas aparecen líneas así con tanto soporte como el borde (sección 6). Si ninguna línea fuerte la lleva, se busca
   la línea oscura directamente (3.7).
3. **Lado de la platina.** Entre las que quedan hay que elegir una, y aquí entra lo único que se declara. Un bloque
   rodeado de platina tiene dos bordes válidos: en el izquierdo la platina queda a la izquierda, y en el derecho, a la
   derecha. La fase no dice de qué lado está la platina: con cualquiera de las dos lecturas la fase es la misma salvo
   el signo, y declarar mal el lado se lee $1 - F$ (cuaderno `20261027-paridad-de-la-lectura`). Por eso el lado se
   declara (`--platen-side`, o `lado_platina` en el sidecar; izquierda por defecto) y **se toma la candidata más
   cercana a ese lado**.

**Cómo leer la figura.** El mapa de saltos $S$ con los picos de cada banda (puntos blancos), las líneas fuertes
(naranja) y la elegida como borde (rojo grueso). En el banco solo se ve un borde, así que hay una sola línea fuerte. Los
puntos del borde se concentran en las filas centrales (por lo explicado arriba); los sueltos, como los de x ≈ 1140
arriba o x ≈ 1510 abajo, son los intrusos que RANSAC dejó fuera. Debajo se imprime, para cada línea fuerte, la
profundidad de la línea oscura a lo largo de ella (umbral 0,25).

In [ ]:
strong_lines = sorted((line for line in lines if line[2] >= geo_mod.STRONG_FRACTION * best_support),
                      key=lambda line: line[0] + line[1] * y_ref)
period = 1.0 / np.hypot(*phase_map.carrier)
reach = max(geo_mod.REFINE_PX, geo_mod.REFINE_PERIODS * period)
half_rows = int(min(geo_mod.DARK_FOLLOW_ROWS, max(200, 3 * period)))
rows_d = (max(0, y_e - half_rows), min(height, y_e + half_rows))
depths = [geo_mod._dark_line_across(img, a, b, rows_d, reach)[1] for a, b, _ in strong_lines]
confirmed = [line for line, d in zip(strong_lines, depths) if d >= geo_mod.DARK_LINE_DEPTH]
chosen = (confirmed or strong_lines)[0]       # platen declared on the left
for (a, b, sup), d in zip(strong_lines, depths):
    print(f"strong line at x = {a + b * y_e:7.1f} px ({100 * sup:.0f} % of the bands): dark line depth {d:.2f}")
yy = np.array([y_lo, y_hi])
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.imshow(step, cmap="viridis")
ax.plot(pts[:, 1], pts[:, 0], "w.", ms=3, label="picos por banda")
for a, b, _ in strong_lines:
    ax.plot(a + b * yy, yy, color="orange", lw=1.5)
ax.plot(chosen[0] + chosen[1] * yy, yy, color="red", lw=2.5, label="borde elegido (platina a la izquierda)")
ax.plot([], [], color="orange", label="líneas fuertes")
ax.set_xlabel("columna x (px)")
ax.set_ylabel("fila y (px)")
ax.set_title("líneas de RANSAC sobre el mapa de saltos")
ax.legend(fontsize=7, loc="lower right")
fig.tight_layout()

### 3.7 La línea oscura del chaflán: confirmación y afinado

**Qué es.** El chaflán del bloque refleja la luz fuera de la apertura y se ve como una **línea oscura** de unos 10 px
a lo largo del borde. Es un detalle físico fijo: no se mueve cuando las franjas se mueven o se inclinan, y está aunque
haya pocas franjas. Sirve para dos cosas:

- **Confirmar** que una línea de saltos de fase es el borde (3.6). Una línea espuria de la demodulación no la tiene.
- **Afinar** la posición. La línea de saltos está suavizada (ventana de 5 px y filtro de Fourier) y se aleja del borde
  cuando las franjas se ensanchan: 7 px y unos 2° con un periodo de 133 px en el banco.

**Cómo se mide** (`_dark_line_across`):

1. Se muestrea la imagen **a lo largo de la línea** candidata $x = a + b\,y$ (con interpolación lineal, en cualquier
   ángulo) y se promedia por distancia a la línea. Las filas son $y_e \pm 200$, o $\pm 3$ periodos si es más (con un
   máximo de $\pm 400$), para promediar al menos tres franjas. Se suaviza en 3 px.
2. Se busca el mínimo dentro del **alcance**: 4 px, o 0,15 periodos si es más. Debe ser un mínimo local; si el perfil
   sigue bajando en el extremo del alcance, la línea oscura no está ahí.
3. Su **profundidad** es $1 - I_\text{mín} / \bar I_\text{anillo}$, donde el anillo es el perfil entre 10 y 20 px del
   mínimo **del lado más oscuro**. Con el lado más oscuro, un escalón de iluminación (un lado claro y otro oscuro) no
   cuenta como línea. Se acepta con una profundidad de al menos **0,25**. En la campaña los bordes del bloque dan de
   0,41 a 0,53, los bordes del haz de 0,11 a 0,16 y las líneas espurias menos de 0,06.

**Ángulo** (`_dark_line_slope`). Se mide el centro de la línea oscura en bandas de 100 filas a lo largo de
$y_e \pm 400$ y se ajusta una recta. Las franjas mueven esos centros unos 3 px de banda a banda, así que con franjas
densas el ángulo de la fase es más estable; solo se usa el de la línea oscura si contradice al de la fase por más de 3
errores típicos.

**Si ninguna línea de fase la lleva** (`_dark_columns`), se buscan los mínimos de las medias por columna dentro de la
zona (lejos de sus bordes) con la misma profundidad, y el borde es el más cercano al lado de la platina. Supone un borde
casi vertical. **Si tampoco hay línea oscura**, el borde queda en el salto de fase.

El resultado dice de dónde salió el borde (`edge_source`): `"phase step and dark line"`, `"dark line"` o
`"phase step"` (sin confirmar; `run_calibration` avisa).

**Cómo leer la figura.** El perfil de intensidad a través del borde en la imagen normalizada, donde el borde ya es
vertical. La línea azul discontinua es el borde de fase (3.6); la naranja, el borde final, sobre la línea oscura. Las
franjas grises son el anillo a cada lado; la profundidad se mide contra el más oscuro de los dos.

In [ ]:
geo = locate_block(img, eval_row=y_e)
col_means = np.convolve(geo.image[rows_d[0]:rows_d[1]].mean(axis=0), np.ones(3) / 3, "same")
c = int(round(geo.edge_x))
print(f"edge source: {geo.edge_source}; phase-step edge {geo.edge_x_phase:.2f} px; final edge {geo.edge_x:.2f} px; "
      f"dark line depth {geo.dark_line_depth:.2f} (threshold {geo_mod.DARK_LINE_DEPTH}); rows {rows_d}; "
      f"reach {reach:.1f} px")

xz = np.arange(c - 40, c + 41)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(xz, col_means[xz], "k.-", lw=0.8, ms=3)
ax.axvspan(c - 20, c - 10, color="0.85")
ax.axvspan(c + 11, c + 21, color="0.85", label="anillo de referencia (10-20 px)")
ax.axvline(geo.edge_x_phase, color="tab:blue", ls="--", label=f"borde de fase {geo.edge_x_phase:.2f}")
ax.axvline(geo.edge_x, color="tab:orange", lw=1.5, label=f"borde final {geo.edge_x:.2f}")
ax.set_xlabel("columna x (px)")
ax.set_ylabel("intensidad media (cuentas)")
ax.set_title(f"línea oscura en el borde: profundidad {geo.dark_line_depth:.2f}")
ax.legend(fontsize=7)
fig.tight_layout()

### 3.8 Filas de la cara

**Por qué hacen falta.** La transformada de Fourier por columnas usa las filas $y_e \pm 450$. Si un extremo del bloque
(arriba o abajo) cae dentro de esas filas, la columna mezcla cara y platina y la fase se contamina. Hay que saber entre
qué filas tiene franjas la cara.

**Cómo se obtienen.**

1. Se toma una **tira de la cara**, de 60 a 160 px a la derecha del borde.
2. Se marcan las filas en que la tira está casi entera (más del 90 %) dentro de la zona con franjas, y el tramo se
   extiende hacia arriba y hacia abajo desde $y_e$ mientras se cumpla.
3. Se promedia el mapa de saltos $S$ a lo ancho de la tira, fila por fila. Un **extremo del bloque** es una línea
   horizontal de saltos, que aparece como un pico de ese perfil. Se aceptan picos con prominencia de al menos el mayor
   entre el 30 % del máximo y 6 veces la desviación absoluta mediana del perfil (una medida del ruido que no se deja
   arrastrar por los picos).
4. La cara útil termina **un periodo de franja antes** del pico, donde la demodulación ya no está perturbada por el
   salto.

**Cómo leer la figura.**

- **Izquierda:** la imagen normalizada con la tira de la cara (recuadro cian), el borde (naranja) y los límites de las
  filas de la cara (rojo).
- **Derecha:** el perfil de $S$ en la tira a lo largo de $y$. El sombreado verde son las filas en que la tira está dentro
  de la zona; las líneas rojas, el tramo de la cara; la punteada gris, la fila de evaluación. Si el bloque termina
  dentro de la zona, su extremo es un pico del perfil; si no, el tramo lo cierra el borde de la zona.

**En esta imagen** ningún extremo del bloque cae dentro de la zona: el perfil es casi plano entre las filas 627 y 1660.
El límite de arriba (627) es simplemente donde la tira sale de la zona. El de abajo sale de un «pico» en la fila 1660,
que no es un extremo del bloque: es el final de la zona, donde el perfil, puesto a cero fuera de ella, cae de golpe y
deja el último valor como pico. Al restarle un periodo (37 px) queda 1624. Es una asimetría del código: abajo se resta
el periodo y arriba no. No cambia la medición, porque las filas de Fourier se recortan además a $y_e \pm 450$ (quedan 627–1500), pero
conviene saberlo.

In [ ]:
x0, x1 = int(geo.edge_x + 60), int(min(width, geo.edge_x + 160))
_, zone_n, step_n, *_ = geo_mod._detect(geo.image, "left", None)
inside = zone_n[:, x0:x1].mean(axis=1) > 0.9
strip_profile = uniform_filter1d(step_n[:, x0:x1].mean(axis=1), 9)
print(f"face rows {geo.face_rows}; Fourier rows used by run_calibration: "
      f"{max(geo.face_rows[0], y_e - 450)}-{min(geo.face_rows[1], y_e + 450)}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [1, 1.2]})
ax[0].imshow(geo.image)
ax[0].add_patch(plt.Rectangle((x0, 0), x1 - x0, height, fill=False, ec="cyan", lw=1))
for yf in geo.face_rows:
    ax[0].axhline(yf, color="red", lw=1)
ax[0].axvline(geo.edge_x, color="orange", lw=1)
ax[0].set_title("tira de la cara (cian) y filas de la cara (rojo)")
ax[0].set_xlabel("columna x (px)")
ax[0].set_ylabel("fila y (px)")
ax[1].plot(strip_profile, color="k", lw=0.8)
ax[1].fill_between(np.arange(height), 0, strip_profile.max() * inside, color="lime", alpha=0.2,
                   label="tira dentro de la zona")
for yf in geo.face_rows:
    ax[1].axvline(yf, color="red", lw=1)
ax[1].axvline(y_e, color="0.4", ls=":", label="fila de evaluación")
ax[1].set_xlabel("fila y (px)")
ax[1].set_ylabel("S medio en la tira (rad)")
ax[1].set_title("perfil de saltos a lo largo de y")
ax[1].legend(fontsize=7)
fig.tight_layout()

### 3.9 El contorno de la cara: borde lejano y extremos

**Por qué hace falta.** Para situar el centro de la cara hay que conocer su contorno: los dos bordes laterales y los
dos extremos. El borde junto a la platina, $x_e$, ya está (3.6–3.7). `locate_block` busca después los otros tres
rasgos en la imagen normalizada.

**Borde lejano** $x_f$ (`_far_edge`). Se busca a la derecha del borde, a más de 80 px, y dentro del campo iluminado.
El orden de preferencia es el mismo que para el borde:

1. una línea fuerte de saltos de fase que lleve la línea oscura del chaflán;
2. si no hay, la línea oscura sola, buscada en el **perfil a lo largo del borde**
   (`_profile_along`). Ese perfil es la media de la imagen sobre rectas paralelas al borde, a $y_e \pm 200$ filas o
   más, y en él las franjas se promedian y la línea oscura queda como un mínimo;
3. si tampoco, una línea de saltos de fase sin confirmar;
4. si no hay ninguna, el **final del campo iluminado** (`_field_border`). Es donde el perfil cae por debajo del 50 % de
   su nivel junto al borde durante al menos 15 px; una línea oscura es más estrecha. **Puede ser el borde del haz y no
   el del bloque**, así que se marca (`"field border"`). Solo se usa con las dimensiones de la cara y si el ancho que
   da concuerda, a menos del 5 %, con un tamaño de píxel declarado (3.10).

**Extremos** $y_t$, $y_b$ (`_face_ends`). Se usan dos tiras:

- una **tira de la cara**, desde 60 px a la derecha del borde hasta 60 px antes del borde lejano;
- una **tira de la platina**, de 100 px de ancho, a 60 px a la izquierda del borde.

Un extremo es una línea horizontal de saltos de fase, que aparece como un pico del perfil de $S$ en la tira de la
cara. Para aceptarlo se piden tres condiciones:

1. **Prominencia** de al menos $0{,}5/\Lambda$ rad. El pico se aplana cuando las franjas se ensanchan, por eso el umbral
   depende del periodo. En el banco los extremos dan de 0,17 a 0,38 rad, con periodos de 37 a 47 px.
2. **La platina no lo comparte.** Dentro de un periodo, la tira de la platina debe saltar menos de la mitad que la de
   la cara. El borde del haz cruza las dos tiras y por eso no pasa: allí la razón es de 0,69 o más; en los extremos del
   banco, de 0,03 a 0,18.
3. **Hay franjas a los dos lados.** Un periodo arriba y otro abajo, la modulación de la tira de la cara es al menos el
   1,5 % de su percentil 95. En los extremos del banco es del 2,5 % o más; en los bordes del haz, en lo oscuro, del
   0,7 %.

Esta búsqueda usa todo el campo con franjas y no solo la zona de 3.2. Por eso encuentra extremos que las **filas de la
cara** de 3.8 no ven: el extremo superior del 2 mm, en la fila 418, queda por encima de la zona, que empieza hacia la
627.

Los umbrales salen de las imágenes de la campaña y de escenas simuladas `[medición propia, 2026-10-29]`.

**Cómo leer la figura** (una por imagen).

- **Izquierda:** la imagen normalizada con el contorno encontrado: el borde (naranja), el borde lejano (magenta) y los
  extremos (rojo). Los recuadros son la tira de la cara (cian) y la de la platina (azul).
- **En medio:** el perfil a lo largo del borde, desde 80 px a su derecha. La línea roja discontinua es el 50 % del
  nivel junto al borde; la gris, el final del campo iluminado; la magenta, el borde lejano.
- **Derecha:** los perfiles de $S$ a lo largo de $y$ en las dos tiras. Los círculos son los picos con la prominencia
  pedida; las líneas rojas, los aceptados como extremos; la punteada, la fila de referencia.

In [ ]:
def outline_panels(g, title):
    # The outline of locate_block (far edge, ends), recomputed step by step on the normalised image.
    im = g.image
    h, w = im.shape
    pm_n, _, st_n, *_ = geo_mod._detect(im, "left", None)
    period_n, period_y = 1.0 / np.hypot(*g.carrier), 1.0 / abs(g.carrier[1])
    a_e, b_e, ye = g.edge_x - g.edge_slope * g.eval_row, g.edge_slope, int(g.eval_row)
    half = int(min(geo_mod.DARK_FOLLOW_ROWS, max(200, 3 * period_n)))
    k, prof = geo_mod._profile_along(im, a_e, b_e, (max(0, ye - half), min(h, ye + half)), 2 * geo_mod.BORDER_MARGIN)
    border = geo_mod._field_border(k, prof)
    level = np.median(prof[:100])
    x0 = int(round(g.edge_x + geo_mod.FACE_INSET))
    x1 = int(round(g.far_edge_x - geo_mod.FACE_INSET if np.isfinite(g.far_edge_x) else g.edge_x + g.face_width_px - 20))
    p1 = int(round(g.edge_x - geo_mod.FACE_INSET))
    p0 = max(0, p1 - 100)
    face_p = uniform_filter1d(st_n[:, x0:x1].mean(axis=1), 9)
    platen_p = uniform_filter1d(st_n[:, p0:p1].mean(axis=1), 9)
    peaks, _ = find_peaks(face_p, prominence=geo_mod.END_PROMINENCE / period_y)

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [0.9, 1.1, 1.3]})
    ax[0].imshow(im)
    ax[0].axvline(g.edge_x, color="orange", lw=1, label="borde (platina)")
    if np.isfinite(g.far_edge_x):
        ax[0].axvline(g.far_edge_x, color="magenta", lw=1, label=f"borde lejano ({g.far_edge_source})")
    for y_end in g.face_ends:
        if np.isfinite(y_end):
            ax[0].axhline(y_end, color="red", lw=1)
    ax[0].plot([], [], color="red", label="extremos")
    ax[0].add_patch(plt.Rectangle((x0, 0), x1 - x0, h, fill=False, ec="cyan", lw=1))
    ax[0].add_patch(plt.Rectangle((p0, 0), p1 - p0, h, fill=False, ec="tab:blue", lw=1))
    ax[0].set_title(title)
    ax[0].legend(fontsize=7, loc="lower left")
    ax[1].plot(g.edge_x + k, prof, color="k", lw=0.8)
    ax[1].axhline(geo_mod.FIELD_BORDER_LEVEL * level, color="tab:red", ls="--", lw=1, label="50 % del nivel junto al borde")
    if np.isfinite(border):
        ax[1].axvline(g.edge_x + border, color="0.5", lw=1.5, label="final del campo iluminado")
    if np.isfinite(g.far_edge_x):
        ax[1].axvline(g.far_edge_x, color="magenta", lw=1, ls=":", label="borde lejano")
    ax[1].set_xlabel("columna x (px)")
    ax[1].set_ylabel("intensidad media a lo largo del borde")
    ax[1].set_title("perfil a lo largo del borde")
    ax[1].legend(fontsize=7)
    ax[2].plot(face_p, color="k", lw=0.8, label="tira de la cara")
    ax[2].plot(platen_p, color="tab:blue", lw=0.8, label="tira de la platina")
    ax[2].plot(peaks, face_p[peaks], "o", mfc="none", mec="0.4", label="picos con la prominencia pedida")
    for y_end in g.face_ends:
        if np.isfinite(y_end):
            ax[2].axvline(y_end, color="red", lw=1)
    ax[2].axvline(g.eval_row, color="0.4", ls=":", lw=1, label="fila de referencia")
    ax[2].set_xlabel("fila y (px)")
    ax[2].set_ylabel("S medio en la tira (rad)")
    ax[2].set_title("extremos: saltos a lo largo de y")
    ax[2].legend(fontsize=7)
    fig.tight_layout()
    plt.show()
    print(f"{title}: far edge {g.far_edge_x:.1f} px ({g.far_edge_source}); ends {g.face_ends[0]:.1f}, "
          f"{g.face_ends[1]:.1f} px; candidate peaks at rows {', '.join(str(p) for p in peaks)}")


outline_panels(geo, "2 mm, repet-2" if HAVE_DATA else "simulado, girado 8°")
if HAVE_DATA:
    img_10 = load_series(DATA / "repetibilidad" / "20261002_L1_tg-inm_10p5mm_repet-1_serie-003", frames=30)
    geo_10 = locate_block(img_10, eval_row=y_e)
    outline_panels(geo_10, "10,5 mm, repet-1 (2 oct.)")

**Lectura.**

- **2 mm.** El borde lejano sale de la **línea oscura sola**, porque más allá apenas hay platina: unos 40 px de
  franjas inclinadas antes de que acabe el campo. No hay línea de fase a lo largo de él. El extremo superior está en la
  fila ≈ 418, con platina por encima. El inferior queda fuera de la imagen. Los otros picos de la tira de la cara son
  el borde del haz arriba (la platina también lo tiene) y el final del campo abajo; los dos se descartan.
- **10,5 mm.** No hay línea oscura más allá del borde: el campo iluminado termina en x ≈ 1638. Ese final se toma como
  borde lejano, **marcado**: con la imagen sola no se sabe si ahí termina el bloque o el haz. La sección 5 muestra que
  esa duda es real: cuando el bloque se movió, el haz lo recortó. El extremo inferior está
  en la fila ≈ 2004, con una tira estrecha de platina debajo. El superior queda fuera del campo; el pico de la fila
  ≈ 78 está en lo oscuro y se descarta por falta de franjas.

En ninguna de las dos imágenes se ve el contorno completo.

### 3.10 El centro de la cara

`locate_face_centre` prueba tres modos, en este orden:

1. **Contorno completo** (`"outline"`). Se ven los dos bordes laterales (sin el final del campo) y los dos extremos.
   El centro es el punto medio: $x_c = (x_e + x_f)/2$, $y_c = (y_t + y_b)/2$. No se da ningún dato.
2. **Contorno parcial y dimensiones** (`"outline and dimensions"`). Se dan las dimensiones de la cara, $a_f \times b_f$
   en mm (`face_mm`; en `run_calibration`, `--face-mm` o `cara_mm` en el sidecar). La escala sale del par de bordes
   laterales, $m_{px} = (x_f - x_e)/a_f$; si no, del par de extremos; si no, del tamaño del píxel declarado. Un eje con
   sus dos rasgos usa el punto medio. Un eje con un solo rasgo toma ese rasgo más la mitad de la dimensión por la escala,
   por ejemplo $y_c = y_t + m_{px}\,b_f/2$. Si se ven los cuatro rasgos, las dimensiones sirven de comprobación: se
   compara el cociente de aspecto medido con $b_f/a_f$.
3. **A mano** (`manual=(y, x)`, en píxeles de la imagen de **entrada**; en `run_calibration`, `--face-centre Y X`).
   Se lleva a la imagen normalizada con el mismo giro y espejo.

Las dimensiones son una propiedad del bloque. Se dan una vez y valen para todas las series y posiciones. Unas
coordenadas a mano solo valen para la toma en que se midieron, y el estudio de posición mostró que el punto tiene que
moverse con el bloque.

**Protección del borde lejano.** Un borde lejano sin línea oscura puede no ser del bloque: el final del campo
iluminado es el borde del haz cuando el haz recorta el bloque, y una línea de saltos de fase puede ser espuria. Con las
dimensiones, el ancho que da, $(x_f - x_e)/a_f$, se compara con el tamaño de píxel declarado (`pixel_mm`; en
`run_calibration`, `--pixel-mm`):

- el final del campo se usa solo si concuerda a menos del 5 %;
- una línea de fase sin confirmar se descarta si no concuerda;
- si no hay tamaño de píxel, el final del campo no se usa.

Sin ese borde, la escala sale de los extremos o del tamaño de píxel. Esta regla se añadió el 2026-10-29, después del
balance de la sección 5.

**Incertidumbre de la posición.** Cada rasgo se toma con 2 px `[sin fuente — hipótesis de trabajo]`; los extremos, con
al menos 0,05 periodos. La demodulación emborrona los extremos a lo largo de las franjas: en lo simulado, el centro se
desvía hasta 3,6 px con periodos de 86 a 128 px. Esas incertidumbres se propagan por las fórmulas de arriba, con las de
las dimensiones (`u_face_mm`). En `run_calibration` pasan a $F$ a través de la cuña:
$u_\text{pos}(F)^2 = (\Delta g_x\,u(x_c))^2 + (\Delta g_y\,u(y_c))^2$, que se suma a $u(F)$.

**Cómo leer la figura.** Cada panel es una imagen normalizada con el contorno usado y la estrella en el centro. En la
escena simulada, el círculo blanco es el centro verdadero. Las dimensiones de los bloques reales son un supuesto: 9 × 30
y 9 × 35 mm. Las caras de los bloques del INM no se han medido. El tamaño de píxel declarado para el 10,5 mm es la
escala del 2 mm, cuyo borde lejano sí confirma la línea oscura, con el ancho supuesto de 9 mm. Se toma así porque es
la misma cámara de la misma campaña, aunque otro día.

In [ ]:
DIMS = {"2mm": (9.0, 30.0), "10p5mm": (9.0, 35.0)}   # assumed faces: not yet measured on the INM blocks
# Declared pixel size: the scale of the 2 mm repet-2, whose far edge the dark line confirms (with the assumed 9 mm).
PIXEL_MM = 1.0 / locate_face_centre(geo, face_mm=DIMS["2mm"]).scale_px_per_mm if HAVE_DATA else None
print(f"declared pixel size: {1000 * PIXEL_MM:.2f} um/px" if PIXEL_MM else "no declared pixel size")

whole = scene(rotation_deg=-5.0, centre_mm=(1.0, 1.5), size_mm=(9.0, 14.0))   # the whole face in the field
g_whole = locate_block(whole.image.astype(float))
h_w, w_w = whole.image.shape
px_mm = 25.0 / (w_w - 1)
true_yx = g_whole.to_normalised((h_w - 1) / 2 + 1.5 / px_mm, (w_w - 1) / 2 + 1.0 / px_mm)
cases = [("simulado: cara de 9 × 14 mm, girada −5°", g_whole, locate_face_centre(g_whole), true_yx)]
if HAVE_DATA:
    for label, g, dims in (("2 mm, repet-2", geo, DIMS["2mm"]), ("10,5 mm, repet-1", geo_10, DIMS["10p5mm"])):
        try:
            locate_face_centre(g)
        except ValueError as exc:
            print(f"{label}, outline only: {exc}")
        cases.append((f"{label}: cara de {dims[0]:g} × {dims[1]:g} mm (supuesta)", g,
                      locate_face_centre(g, face_mm=dims, pixel_mm=PIXEL_MM), None))

fig, ax = plt.subplots(1, len(cases), figsize=(5 * len(cases), 4.6))
for a, (label, g, c, truth_yx) in zip(np.atleast_1d(ax), cases):
    a.imshow(g.image)
    a.axvline(g.edge_x, color="orange", lw=1)
    if np.isfinite(g.far_edge_x):
        a.axvline(g.far_edge_x, color="magenta", lw=1, ls="-" if g.far_edge_source != "field border" else "--")
    for y_end in g.face_ends:
        if np.isfinite(y_end):
            a.axhline(y_end, color="red", lw=1)
    a.plot(c.x, c.y, "*", ms=16, mfc="tab:red", mec="w")
    if truth_yx is not None:
        a.plot(truth_yx[1], truth_yx[0], "o", ms=9, mfc="none", mec="w", mew=1.5)
    a.set_title(f"{label}\n{c.source}: y = {c.y:.1f} ± {c.u_y:.1f}, x = {c.x:.1f} ± {c.u_x:.1f} px", fontsize=8)
    a.set_xticks([]), a.set_yticks([])
fig.tight_layout()
plt.show()

for label, g, c, truth_yx in cases:
    scale = f"; scale {1000 / c.scale_px_per_mm:.2f} um/px ({c.scale_source})" if np.isfinite(c.scale_px_per_mm) else ""
    error = f"; error {c.y - truth_yx[0]:+.2f}, {c.x - truth_yx[1]:+.2f} px" if truth_yx is not None else ""
    print(f"{label}: {c.source}, centre row {c.y:.1f}, column {c.x:.1f} (u {c.u_y:.1f}, {c.u_x:.1f} px){scale}{error}")
    for note in c.notes:
        print(f"    note: {note}")
manual = locate_face_centre(cases[-1][1], manual=(1258.0, 1339.0), u_manual_px=5.0)
print(f"by hand: input (1258, 1339) -> normalised ({manual.y:.1f}, {manual.x:.1f}), source {manual.source}")

**Lectura.**

- **Simulado.** Con el contorno completo, el centro cae a menos de 1 px del verdadero sin dar ningún dato. Las
  pruebas de `interflib` (`tests/test_geometry.py`) cubren además bloques desplazados, platina a la derecha, franjas
  anchas y un solo extremo con las dimensiones `[cálculo propio, 2026-10-29]`. Todos dan 0,3 px o menos en $x$ y
  3,6 px o menos en $y$.
- **2 mm.** Sin dimensiones no hay centro, porque falta el extremo inferior. Con 9 × 30 mm, la escala sale del par de
  bordes, 17,7 µm/px, y el centro queda en la fila ≈ 1263. El 2026-10-20 se había estimado a mano en ≈ 1258.
- **10,5 mm.** Su borde lejano es el final del campo iluminado. Da 18,1 µm/px, a −2,2 % del tamaño de píxel declarado,
  así que se usa. El centro queda en la fila ≈ 1040.

**Límite hallado en lo simulado.** Con $F$ entre 0,4 y 0,5 y sin línea oscura (el simulador no la dibuja), cada borde
se parte en dos líneas a ±16 px y el centro se desvía de 7 a 12 px en $x$. Viene de antes de este cambio y sigue
abierto. En el banco, la línea oscura afina el borde de la platina.

### 3.11 Las áreas resultantes

Con el borde, el centro de la cara y las filas de la cara quedan fijadas todas las áreas. `run_calibration` hace todo
lo anterior por sí solo. Basta con darle la serie, la presión y $\lambda_0$, porque el sidecar no trae la presión, y las
dimensiones de la cara. Estas van en `--face-mm` o, en las series nuevas, en `cara_mm` del sidecar. Aquí se suponen 75,3
kPa, 543,5163 nm, una cara de 9 × 30 mm y 17,74 µm/px.

**Cómo se lee $F$ en el centro.** La fila de evaluación pasa por el centro. Allí la cara del bloque está en la imagen y
la platina no, porque el bloque la tapa. Por eso:

- la fase del **bloque** se mide en el centro, con bandas de $x_c - 75$ a $x_c + 75$ px, y su recta se interpola en
  $x_c$;
- solo la recta de la **platina**, ajustada a sus bandas de 30 a 180 px del borde, se prolonga hasta $x_c$, unos 280
  px.

Las dos fases se comparan así en el mismo píxel, y la inclinación de la referencia se sigue cancelando. Lo que queda es
suponer la platina recta en la prolongación. Para cubrirlo, $u(F)$ suma una cota de tipo B,
$u_\text{ext}(F) = 0{,}065$ franjas, que es lo que falló la recta de la platina frente a la platina visible sobre el 2 mm
`[medición propia, campaña 202610]`.

La figura se lee como la de la sección 1, con estas diferencias:
- la estrella marca el centro en (a);
- en (b) el recuadro rojo de las bandas del bloque está alrededor del centro (línea blanca de trazos y puntos);
- en (c) la recta de la platina llega hasta el centro y la del bloque pasa por sus bandas.

In [ ]:
if HAVE_DATA:
    real = show_areas(["--series", str(REPET), "--pressure", "75300", "--wavelength-vac", "543.5163",
                       "--face-mm", "9", "30", "--pixel-mm", "0.01774"], "real-2mm")
    frac_r, centre_r = real["fraction"], real["image"]["face_centre"]
    print(f"detection: {real['image']['geometry']['method']}; edge {real['image']['edge_x_px']:.2f} px "
          f"({real['image']['edge_origin']})")
    print(f"face centre ({centre_r['source']}): row {centre_r['y']:.1f}, column {centre_r['x']:.1f}; "
          f"evaluation row {real['image']['eval_row_px']}")
    print(f"F at the edge = {frac_r['edge_fraction']:.4f}; at the face centre, block read there, "
          f"{frac_r['fraction_used']:.4f}")
    print(f"platen line extended {frac_r['platen_extension_px']:.0f} px; u_e(F) = {frac_r['u_extrapolation']:.4f}, "
          f"u_m(F) = {frac_r['u_band_choice']:.4f}, u_pos(F) = {frac_r['u_centre_position']:.4f}, "
          f"u_ext(F) = {frac_r['u_platen_extension']:.3f}; u(F) = {frac_r['u_fraction']:.4f} fringe")
else:
    print("no campaign data: skipped")

## 4. Bloque girado y platina a la derecha: giro y espejo

El banco no tiene bloques girados, así que esta parte se muestra con la escena simulada: el bloque girado 8° y
desplazado 2 mm, rodeado de platina.

**Giro.** El ángulo del borde es $\theta = \arctan b$, con $b$ la pendiente de la recta de RANSAC. Si $|\theta| \ge 1°$,
la imagen se gira $\theta$ alrededor de su centro con interpolación cúbica (`scipy.ndimage.affine_transform`), para que
el borde quede vertical. Por debajo de 1° no se gira, para no interpolar: con 1° el borde se corre 2,6 px en las 150
filas de la ventana, dentro del hueco de 30 px que dejan los lectores junto al borde.

**Espejo.** Si la platina se declara a la derecha, la imagen se refleja en $x$ para que los lectores la vean a la
izquierda.

Después de girar o reflejar se repite la detección sobre la imagen normalizada, para tener el borde en sus nuevas
coordenadas.

**Cómo leer la figura.**

- **Arriba a la izquierda:** la escena de entrada. El bloque es la franja central, con las franjas desfasadas respecto
  a la platina que lo rodea.
- **Arriba a la derecha:** su mapa de saltos con las dos líneas fuertes de RANSAC (naranja), una por cada borde del
  bloque.
- **Abajo:** las dos imágenes normalizadas, declarando la platina a la izquierda (se toma el borde izquierdo y se gira)
  y a la derecha (se toma el derecho, se gira y se refleja). En las dos, la línea naranja es el borde y los recuadros son
  las bandas de platina (azul) y de bloque (rojo) en la ventana de ajuste. Las esquinas sin imagen las rellena el giro
  con el valor del borde más cercano.

In [ ]:
img_t = tilted.image.astype(float)
h_t, w_t = img_t.shape
pm_t, zone_t = fringe_zone(img_t)
step_t = phase_step_map(pm_t)
*_, y_ref_t, lines_t = geo_mod._detect(img_t, "left", None)
truth = tilted.fraction_truth

fig, ax = plt.subplots(2, 2, figsize=(11, 8))
ax[0, 0].imshow(img_t)
ax[0, 0].set_title("entrada: bloque girado 8°, rodeado de platina")
ax[0, 1].imshow(step_t, cmap="viridis")
yy = np.array([0, h_t])
for a, b, _ in lines_t:
    ax[0, 1].plot(a + b * yy, yy, color="orange", lw=1.5)
ax[0, 1].set_xlim(0, w_t)
ax[0, 1].set_ylim(h_t, 0)
ax[0, 1].set_title("mapa de saltos y las dos líneas fuertes")
for k, side in enumerate(("left", "right")):
    g = locate_block(img_t, platen_side=side)
    ye = h_t // 2
    rows = (ye - 75, ye + 75)
    fft_rows = (max(g.face_rows[0], ye - 450), min(g.face_rows[1], ye + 450))
    F = fourier_edge_fraction(g.image, g.edge_x, rows, ye, fft_rows=fft_rows).fraction
    a = ax[1, k]
    a.imshow(g.image)
    a.axvline(g.edge_x, color="orange", lw=1.2)
    a.add_patch(plt.Rectangle((g.edge_x - 180, ye - 75), 150, 150, fill=False, ec="tab:blue", lw=1.5))
    a.add_patch(plt.Rectangle((g.edge_x + 30, ye - 75), 150, 150, fill=False, ec="tab:red", lw=1.5))
    a.set_title(f"platina declarada a la {'izquierda' if side == 'left' else 'derecha'}: girada "
                f"{g.rotation_deg:+.1f}°{', reflejada' if g.mirrored else ''}\nF = {F:.4f} (verdad {truth:.4f})")
for a in ax.ravel():
    a.set_xticks([]), a.set_yticks([])
fig.tight_layout()

Las dos declaraciones toman bordes distintos del mismo bloque. Como en los dos la platina está efectivamente del lado
declarado, las dos dan la misma $F$, igual a la verdad. La heurística del banco, en esta escena, se equivocaba en
104 px (sección 2).

## 5. Balance en la campaña 2026-10

Se aplica `locate_block` a las 22 series de análisis (densidad de franjas, posición y repetibilidad), con la media de
los 5 primeros cuadros de cada una, y se compara con la heurística del banco. En estas series la geometría del banco se
cumple, así que la heurística sirve de referencia. Tarda unos minutos.

Columnas de la tabla: borde de la heurística y automático (px), su diferencia, de dónde salió el borde
(`edge_source`: `phase+dark`, `dark` o `phase`), la profundidad de la línea oscura, el ángulo del borde, su soporte
(fracción de bandas; vacío si el borde es la línea oscura sola) y las anchuras de platina y cara en la fila de
evaluación. El centro de la cara se tabula aparte, más abajo, y al final una galería muestra lo detectado sobre
cada interferograma.

In [ ]:
SOURCE_LABEL = {"phase step and dark line": "phase+dark", "dark line": "dark", "phase step": "phase"}
THUMB = 4                     # thumbnails for the gallery: one pixel in four along each axis
results = []
if HAVE_DATA:
    folders = sorted(p for p in DATA.glob("*/*") if p.is_dir() and "_serie-" in p.name
                     and ("20261005" in p.name or "repet" in p.name))
    for folder in folders:
        frame = load_series(folder)
        x_ref, _ = bench_heuristic(frame, run_calibration.BENCH_EDGE_ROWS)
        t0 = time.perf_counter()
        try:
            g = locate_block(frame, eval_row=run_calibration.BENCH_EVAL_ROW)
            row = dict(edge=g.edge_x, diff=g.edge_x - x_ref, source=SOURCE_LABEL[g.edge_source],
                       depth=g.dark_line_depth, angle=g.angle_deg, support=g.support, platen=g.platen_width_px,
                       face=g.face_width_px, far=g.far_edge_x, far_source=g.far_edge_source, top=g.face_ends[0],
                       bottom=g.face_ends[1], slope=g.edge_slope, eval_row=g.eval_row,
                       thumb=g.image[::THUMB, ::THUMB].astype(np.float32), error=None)
            try:
                c = locate_face_centre(g, face_mm=DIMS["2mm" if "_2mm_" in folder.name else "10p5mm"],
                                       pixel_mm=PIXEL_MM)
                row.update(yc=c.y, xc=c.x, u_yc=c.u_y, um_px=1000 / c.scale_px_per_mm, scale=c.scale_source,
                           far_used=not any("not used" in note for note in c.notes), centre_error=None)
            except ValueError as exc:
                row.update(yc=np.nan, xc=np.nan, u_yc=np.nan, um_px=np.nan, centre_error=str(exc))
        except ValueError as exc:
            row = dict(edge=np.nan, diff=np.nan, source="-", depth=np.nan, angle=np.nan, support=np.nan,
                       platen=np.nan, face=np.nan, thumb=frame[::THUMB, ::THUMB].astype(np.float32),
                       error=str(exc))
        row.update(name=folder.name.split("_tg-inm_")[1].split("_serie")[0], heuristic=x_ref,
                   seconds=time.perf_counter() - t0)
        results.append(row)

    print(f"{'series':18s} {'heuristic':>9s} {'auto':>8s} {'diff':>7s} {'source':>10s} {'depth':>5s} {'angle':>7s} "
          f"{'support':>7s} {'platen':>6s} {'face':>5s}")
    for r in results:
        if r["error"]:
            print(f"{r['name']:18s} {r['heuristic']:9.1f}   FAILED: {r['error']}")
            continue
        flag = "  <-- wrong" if abs(r["diff"]) > 5 else ""
        support = f"{100 * r['support']:6.0f}%" if np.isfinite(r["support"]) else " " * 7
        print(f"{r['name']:18s} {r['heuristic']:9.1f} {r['edge']:8.1f} {r['diff']:+7.1f} {r['source']:>10s} "
              f"{r['depth']:5.2f} {r['angle']:+7.2f} {support} {r['platen']:6.0f} {r['face']:5.0f}{flag}")
    print(f"\nmean time per image: {np.mean([r['seconds'] for r in results]):.1f} s")
else:
    print("no campaign data: skipped")

**Cómo leer la figura.** Cada columna es una serie, con su bloque, su estudio y la densidad de franjas en el nombre
(`dens-20f`: unas 20 franjas en el campo). El eje vertical es la diferencia absoluta entre el borde automático y el de
la heurística, en escala logarítmica. Círculo verde: salto de fase confirmado por la línea oscura. Cuadrado azul: la
línea oscura sola. Cruz roja: salto de fase sin confirmar (no aparece en la campaña). Triángulo gris (dibujado arriba,
fuera de escala): `locate_block` falló con un error, y `run_calibration` usaría la heurística. La línea punteada marca
5 px: por encima, el borde automático está mal.

In [ ]:
if results:
    fig, ax = plt.subplots(figsize=(11, 3.6))
    for i, r in enumerate(results):
        if r["error"]:
            ax.plot(i, 300, "v", color="0.5", ms=9)
            continue
        marker, color = {"phase+dark": ("o", "tab:green"), "dark": ("s", "tab:blue")}.get(r["source"], ("x", "tab:red"))
        ax.plot(i, max(abs(r["diff"]), 0.02), marker, color=color, ms=8)
    ax.set_yscale("log")
    ax.set_ylim(0.01, 600)
    ax.axhline(5, color="0.6", ls=":", lw=1)
    ax.set_xticks(range(len(results)), [r["name"] for r in results], rotation=60, ha="right")
    ax.set_ylabel("|auto − heurística| (px)")
    ax.plot([], [], "o", color="tab:green", label="salto de fase + línea oscura")
    ax.plot([], [], "s", color="tab:blue", label="línea oscura sola")
    ax.plot([], [], "x", color="tab:red", label="salto de fase sin confirmar")
    ax.plot([], [], "v", color="0.5", label="falla (ValueError)")
    ax.legend(fontsize=8, loc="upper right")
    ax.set_title("borde automático frente a la heurística del banco")
    fig.tight_layout()

**Lectura del balance** `[medición propia, 2026-10-28]`:

- **Ningún borde equivocado.** Las 19 series que se demodulan caen a 2,2 px o menos de la heurística.
- **16 series** (20, 40 y 80 franjas, posición y repetibilidad): salto de fase confirmado por la línea oscura, de −2,2 a
  +0,5 px.
- **3 series** (10,5 mm con 10 franjas, 2 mm con 5 y con 10): ninguna línea de fase lleva la línea oscura, y el borde
  es la línea oscura sola, de −0,8 a +0,2 px.
- **Con 3 franjas, y con 5 en el 10,5 mm**, la estimación de la portadora no encuentra 4 franjas en el recorte (3.1) y
  `locate_block` falla con un error. `run_calibration` vuelve entonces a la heurística del banco y lo avisa. Es un fallo
  **detectado**.

**Antes de la confirmación** (versión del 2026-10-27, la primera de este cuaderno), con 5 a 20 franjas había **fallos
silenciosos** en 4 de 6 series: −7,9 px (10,5 mm, 10 franjas), −149 px (10,5 mm, 20), −64 px y −27° (2 mm, 5) y
−43 px (2 mm, 10). En todos el borde había quedado sin afinar por la línea oscura, y eso llevó a usarla como
confirmación. El 2 mm movido a la derecha quedaba a 3,7 px porque la línea oscura estaba a 4,0 px, justo fuera del
alcance de entonces; ahora queda a −0,2 px.

**Centro de la cara en la campaña.** En la misma pasada se sitúa el centro con las caras supuestas de 9 × 30 mm
(2 mm) y 9 × 35 mm (10,5 mm). Columnas de la tabla:

- el origen del borde lejano (`dark`: línea oscura; `phase+dark`: línea de fase con línea oscura; `field`: final del
  campo iluminado; `none`);
- los extremos encontrados, en filas;
- el centro $(y_c, x_c)$ con la incertidumbre de $y_c$;
- la distancia del centro al borde, $x_c - x_e$;
- la escala en µm/px y de dónde sale (`width`: par de bordes; `pixel`: tamaño de píxel declarado; `length`: par de
  extremos). Un asterisco en el borde lejano indica que no se usó (protección de 3.10).

En el estudio de posición el bloque se movió, así que el centro tiene que moverse con él.

In [ ]:
FAR_LABEL = {"phase step and dark line": "phase+dark", "dark line": "dark", "phase step": "phase",
             "field border": "field", "none": "none"}
SCALE_LABEL = {"face width": "width", "face length": "length", "pixel size": "pixel"}
if results:
    print(f"{'series':18s} {'far edge':>15s} {'top':>7s} {'bottom':>7s} {'y_c':>7s} {'u(y_c)':>6s} {'x_c - x_e':>9s} "
          f"{'um/px':>6s} {'scale':>6s}")
    for r in results:
        if r["error"]:
            print(f"{r['name']:18s}   no block: {r['error'][:60]}")
            continue
        mark = "*" if not r.get("far_used", True) else " "
        far = (f"{r['far']:7.1f} {FAR_LABEL[r['far_source']]:>6s}{mark}" if np.isfinite(r["far"])
               else f"{'':7s} {'none':>6s} ")
        if r["centre_error"]:
            print(f"{r['name']:18s} {far} {r['top']:7.1f} {r['bottom']:7.1f}   no centre: {r['centre_error'][:50]}")
            continue
        print(f"{r['name']:18s} {far} {r['top']:7.1f} {r['bottom']:7.1f} {r['yc']:7.1f} {r['u_yc']:6.1f} "
              f"{r['xc'] - r['edge']:9.1f} {r['um_px']:6.2f} {SCALE_LABEL[r['scale']]:>6s}")

**Lectura** `[medición propia, 2026-10-29; caras de 9 × 30 y 9 × 35 mm supuestas; tamaño de píxel de 17,74 µm/px
tomado del 2 mm]`:

- **14 de las 19 series que se demodulan tienen centro.** En todas, el centro queda a 247–256 px del borde, la mitad
  del ancho de la cara, y la escala, de 17,7 a 18,2 µm/px.
- **El haz recorta el bloque en tres tomas de posición** (2 mm centro y derecha, 10,5 mm derecha). Allí el final del
  campo iluminado cae dentro de la cara, y la escala que daba era de 19,9 a 30 µm/px. Fue la primera ejecución de esta
  sección la que lo mostró. Desde entonces ese borde se descarta (asterisco), porque no concuerda con el tamaño de
  píxel, y la escala sale del tamaño declarado. Sin esa protección el centro se habría desplazado hasta 100 px en $x$ y
  325 px en $y$ **sin aviso**. En esas tres tomas la incertidumbre de $y_c$ (2 px) no incluye la del tamaño de píxel, que se toma exacto.
- **El centro sigue al bloque.** En el estudio de posición, el 10,5 mm cambia de fila de centro entre tomas (909, 927 y
  989) porque su extremo inferior se movió (1896–1952). Es el registro vertical que se hizo a mano el 2026-10-20.
- **Sin centro, con aviso, en 5 series:** el 2 mm con 5 franjas y el 10,5 mm con 10, 20 y 40 franjas y en `repet-2`. En
  ninguna se encuentra un extremo de la cara. El 10,5 mm tiene apenas 30–40 filas de platina bajo su extremo, y con
  ellas el criterio de franjas a los dos lados queda al límite.
- **Un extremo dudoso: el 2 mm con 10 franjas.** Su extremo superior sale en la fila 370, frente a 418–429 con 20, 40 y
  80 franjas, aunque el bloque no se movió entre esas series. Eso son unos 50 px, cerca de medio periodo, y la
  incertidumbre declarada (7,9 px) no lo cubre. Con franjas anchas, la demodulación desplaza el pico del extremo más de
  lo que se vio en lo simulado. **Queda abierto**, y es un fallo silencioso posible con pocas franjas.

**Galería: lo detectado en cada interferograma.** Las tablas dicen dónde quedó cada rasgo. Aquí se ve si está
donde debe. Cada panel es una serie de la campaña (media de 5 cuadros), ya normalizada (borde vertical, platina a la
izquierda) y reducida a uno de cada 4 píxeles. Las coordenadas de los ejes son las de la imagen completa.

**Cómo leer cada panel.**

- **Naranja:** el borde junto a la platina, $x_e$.
- **Magenta continuo:** el borde lejano usado para el centro. **Magenta discontinuo:** el borde lejano encontrado
  pero no usado: o no concuerda con el tamaño de píxel (el asterisco de la tabla), o la serie no tiene centro.
- **Rojo horizontal:** los extremos de la cara encontrados, entre los dos bordes.
- **Estrella roja:** el centro de la cara. **Línea blanca punteada:** la fila de referencia (1050), donde se leía $F$
  antes.
- **Título:** la serie y el origen del borde. Debajo, el modo del centro y la escala, o por qué no hay centro. Los
  paneles con fondo rojo son las series en que `locate_block` no encuentra el bloque; muestran la imagen sin
  normalizar.

In [ ]:
if results:
    n_cols = 4
    n_rows = int(np.ceil(len(results) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.9 * n_rows))
    for a, r in zip(axes.ravel(), results):
        thumb = r["thumb"]
        h_t, w_t = thumb.shape
        extent = (0, w_t * THUMB, h_t * THUMB, 0)                    # axes in full-image pixels
        a.imshow(thumb, extent=extent, vmin=0, vmax=np.percentile(thumb, 99.5))
        a.set_xticks([]), a.set_yticks([])
        if r["error"]:
            a.set_facecolor("mistyrose")
            for spine in a.spines.values():
                spine.set_color("tab:red"), spine.set_linewidth(2)
            a.set_title(f"{r['name']}\nsin bloque: pocas franjas", fontsize=8, color="tab:red")
            continue
        yy = np.array([0, h_t * THUMB])
        a.plot(r["edge"] + r["slope"] * (yy - r["eval_row"]), yy, color="orange", lw=1)
        if np.isfinite(r["far"]):
            used = r.get("far_used", True) and r["centre_error"] is None
            a.plot(r["far"] + r["slope"] * (yy - r["eval_row"]), yy, color="magenta", lw=1,
                   ls="-" if used else "--")
        x_right = r["far"] if np.isfinite(r["far"]) else r["edge"] + r["face"]
        for y_end in (r["top"], r["bottom"]):
            if np.isfinite(y_end):
                a.plot([r["edge"], x_right], [y_end, y_end], color="red", lw=1.2)
        a.axhline(r["eval_row"], color="w", ls=":", lw=0.8)
        if r["centre_error"] is None:
            a.plot(r["xc"], r["yc"], "*", ms=13, mfc="red", mec="w", mew=0.8)
            note = f"centro: {SCALE_LABEL[r['scale']]}, {r['um_px']:.2f} µm/px"
        else:
            note = "sin centro: " + ("ningún extremo" if "no end" in r["centre_error"] else r["centre_error"][:30])
        a.set_xlim(0, w_t * THUMB)
        a.set_ylim(h_t * THUMB, 0)
        a.set_title(f"{r['name']} · borde {r['source']}\n{note}", fontsize=8)
    for a in axes.ravel()[len(results):]:
        a.axis("off")
    fig.tight_layout()
    plt.show()
else:
    print("no campaign data: skipped")

**Lectura de la galería** `[medición propia, 2026-10-29]`:

- **El borde junto a la platina** (naranja) cae sobre el canto izquierdo de la cara en las 19 series que se demodulan,
  con 5 a 80 franjas y en las tres posiciones del bloque.
- **El borde lejano del 2 mm** (magenta continuo) cae sobre su línea oscura en las series de densidad, de
  repetibilidad y en la toma de la izquierda.
- **El borde lejano del 10,5 mm** es el final del campo iluminado. En las series de densidad y de repetibilidad
  coincide a la vista con el final de las franjas de la cara.
- **El haz recorta el bloque** en el 2 mm en el centro y a la derecha, y en el 10,5 mm a la derecha. Ahí el final del
  campo (discontinuo) corta la cara. El centro se sitúa entonces con el tamaño de píxel declarado y queda a media cara
  del borde, no a medio camino entre las dos líneas.
- **10,5 mm con 10 franjas.** La línea discontinua cae **dentro** de la cara, a unos 300 px del borde: es una línea de
  saltos de fase espuria. Aquí no se usa porque la serie no tiene ningún extremo. Si lo tuviera, el ancho que da
  (≈ 29 µm/px) estaría muy lejos del tamaño de píxel y se descartaría.
- **Extremos.**
  - El superior del 2 mm aparece en todas sus series con 10 franjas o más.
  - El inferior del 10,5 mm solo aparece en 5 de sus 9 series con bloque, porque bajo él quedan pocas filas de
    platina.
  - En el 2 mm con 10 franjas el extremo queda visiblemente por encima del de las demás densidades (fila 370 frente a
    418–429), el desvío anotado arriba.
- **Las tres series sin bloque** (recuadro rojo, 3 y 5 franjas) muestran por qué fallan: hay demasiado pocas franjas
  para la demodulación.

## 6. Por qué fallaba y cómo se corrigió

Se repiten los pasos de la sección 3 en las dos series que mejor muestran los fallos de la versión anterior.

**Cómo leer cada figura (cuatro paneles).**

1. La imagen media y el contorno de la zona con franjas (3.2).
2. La modulación $b$.
3. La fase sin portadora $\psi$ (3.3).
4. El mapa de saltos $S$ con los picos de cada banda (puntos blancos), las líneas fuertes de RANSAC (naranja), la
   que elegía la regla anterior (rojo grueso), el borde que se elige ahora (verde claro) y, en cian discontinuo, el
   borde de la heurística, que aquí es el verdadero.

In [ ]:
def stages(img_s, title):
    pm, zn = fringe_zone(img_s)
    st = phase_step_map(pm)
    *_, y_mid, strong = geo_mod._detect(img_s, "left", None)
    y_eval = run_calibration.BENCH_EVAL_ROW
    strong = sorted(strong, key=lambda line: line[0] + line[1] * y_mid)
    rows_f = np.flatnonzero(zn.any(axis=1))
    lo_r, hi_r = rows_f.min(), rows_f.max()
    bands = np.arange(lo_r, max(lo_r + 1, hi_r - geo_mod.SLAB_ROWS), geo_mod.SLAB_ROWS)
    pk = np.array([((b0 + b0 + geo_mod.SLAB_ROWS) / 2, x) for b0 in bands
                   for x, _ in geo_mod._interior_peaks(st, zn, b0, min(b0 + geo_mod.SLAB_ROWS, img_s.shape[0]))])
    x_h, _ = bench_heuristic(img_s, run_calibration.BENCH_EDGE_ROWS)
    fig, ax = plt.subplots(1, 4, figsize=(15, 3.8))
    ax[0].imshow(img_s)
    ax[0].contour(zn, [0.5], colors="lime", linewidths=0.8)
    ax[0].set_title("1. imagen y zona")
    ax[1].imshow(pm.modulation, cmap="magma")
    ax[1].set_title("2. modulación b")
    ax[2].imshow(pm.object_phase, cmap="twilight")
    ax[2].set_title("3. fase sin portadora ψ")
    ax[3].imshow(st, cmap="viridis")
    yy = np.array([lo_r, hi_r])
    if len(pk):
        ax[3].plot(pk[:, 1], pk[:, 0], "w.", ms=2)
    for a, b, _ in strong:
        ax[3].plot(a + b * yy, yy, color="orange", lw=1.5)
    ax[3].plot(strong[0][0] + strong[0][1] * yy, yy, color="red", lw=2.5)
    period_s = 1.0 / np.hypot(*pm.carrier)
    _, (a_new, b_new), depth_new, source = geo_mod._choose_edge(img_s, zn, strong, "left", y_eval, period_s)
    ax[3].plot(a_new + b_new * yy, yy, color="lime", lw=1.5)
    ax[3].axvline(x_h, color="cyan", ls="--", lw=1)
    ax[3].set_xlim(0, img_s.shape[1])
    ax[3].set_ylim(img_s.shape[0], 0)
    ax[3].set_title("4. saltos, picos y líneas")
    for a in ax:
        a.set_xticks([]), a.set_yticks([])
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    print(f"{title}: strong lines at x = "
          + ", ".join(f"{a + b * y_mid:.0f} px ({np.degrees(np.arctan(b)):+.1f} deg, {100 * s:.0f} %)"
                      for a, b, s in strong)
          + f"; former rule {strong[0][0] + strong[0][1] * y_mid:.0f} px; bench heuristic {x_h:.0f} px")
    reach_s = max(geo_mod.REFINE_PX, geo_mod.REFINE_PERIODS * period_s)
    half_s = int(min(geo_mod.DARK_FOLLOW_ROWS, max(200, 3 * period_s)))
    rows_s = (y_eval - half_s, y_eval + half_s)
    for a, b, _ in strong:
        print(f"  dark line along the line at {a + b * y_eval:.0f} px: depth "
              f"{geo_mod._dark_line_across(img_s, a, b, rows_s, reach_s)[1]:.3f}")
    print(f"  now: {source}, edge at {a_new + b_new * y_eval:.1f} px (depth {depth_new:.2f})")


if HAVE_DATA:
    stages(load_series(DATA / "densidad-franjas" / "20261005_L1_tg-inm_10p5mm_dens-20f_serie-014"),
           "10,5 mm con 20 franjas")

**10,5 mm con 20 franjas: gana una línea espuria.** RANSAC encuentra dos líneas fuertes con el mismo soporte (59 % de
las bandas): la de x ≈ 1146, que es el borde verdadero (coincide con la heurística), y la de x ≈ 997, dentro de la franja
de platina. La regla de 3.6, «la línea fuerte más cercana al lado de la platina», supone que una línea fuerte del lado
de la platina es el otro borde del bloque, y toma la de 997. Aquí la suposición es falsa. En la modulación (panel 2) se
ve que la zona tiene dos lóbulos separados cerca de x ≈ 1000, y en esa separación la fase también salta.

**Cómo se corrigió.** A lo largo de la línea espuria no hay línea oscura (profundidad cercana a 0); a lo largo del borde,
sí (≈ 0,5). Con la confirmación de 3.6 la espuria ya no compite y se toma el borde (verde claro).

In [ ]:
if HAVE_DATA:
    stages(load_series(DATA / "densidad-franjas" / "20261005_L1_tg-inm_2mm_dens-05f_serie-004"),
           "2 mm con 5 franjas")

**2 mm con 5 franjas: se rompe la demodulación.** Con tan pocas franjas, el lóbulo de la portadora queda muy cerca del
lóbulo del fondo en el espectro (3.1), y el filtro de radio $0{,}5|g|$ no los separa bien. La fase sin portadora (panel
3) deja de ser casi constante en cada superficie y aparece un **vórtice**: un punto alrededor del cual la fase da una
vuelta completa. Las líneas que encuentra RANSAC son artefactos, de −27° y +8°, con solo el 22 % de las bandas, y la
elegida está lejos del borde verdadero. El mínimo actual de 4 franjas en el recorte deja pasar este caso.

**Cómo se corrigió.** Ninguna de las dos líneas lleva la línea oscura, así que ninguna se confirma. La línea oscura sí se
ve en las medias por columna, y el borde pasa a ser ella (verde claro, sobre el cian). Después, con estas 5 franjas, el
lector no puede estimar el periodo (menos de 4 franjas en sus filas): `run_calibration` falla con un error, no en
silencio.

**Lo que queda.** Cuando el borde sale de la línea oscura sola, se toma la más cercana al lado de la platina. Si el
borde cercano no se viera y el lejano sí, se tomaría el lejano, que tiene el bloque del lado de la platina, y se leería
$1 - F$. Pasó una vez con una versión intermedia del método. Como el lado de la platina, solo un valor previo lo
detecta.

## 7. Estado y lo que falta

| Pieza | Método | Estado |
|---|---|---|
| Áreas (bandas, ventana, filas de Fourier) | Derivadas del borde y de la fila de evaluación | **Hecho** |
| Zona con franjas | Umbral de la modulación de Fourier (3.2) | **Hecho** |
| Borde | Mapa de saltos de fase, picos por bandas y RANSAC (3.3–3.5), confirmado por la línea oscura (3.6–3.7) | **Hecho**; correcto en las 19 series de la campaña que se demodulan, y hasta 12° en lo simulado |
| Afinado | Línea oscura del chaflán (3.7) | **Hecho** |
| Orientación | Giro y espejo (4) | **Hecho** |
| Filas de la cara | Picos horizontales de saltos en una tira de la cara (3.8) | **Hecho** |
| Borde lejano | Línea de fase con línea oscura, línea oscura o línea de fase; si no, final del campo iluminado, marcado (3.9) | **Hecho** (2026-10-29) |
| Extremos de la cara | Saltos horizontales en la cara que la platina no comparte, con franjas a los dos lados (3.9) | **Hecho** (2026-10-29) |
| Centro de la cara | Contorno completo; contorno parcial y dimensiones (`--face-mm`, `cara_mm` del sidecar); a mano (3.10) | **Hecho** (2026-10-29); en la campaña solo con las dimensiones, que son un supuesto |
| Pocas franjas (3–5) | Mínimo de 4 franjas en la portadora | **Detectado**: error y vuelta a la heurística |
| 5–20 franjas | Confirmación por la línea oscura (2026-10-28) | **Resuelto**: antes, 4 de 6 fallos silenciosos; ahora ninguno |
| Borde lejano tomado por la línea oscura sola | — | **Riesgo residual**: se leería $1 - F$; solo un valor previo lo detecta |
| Lado de la platina | Se declara | La fase no lo distingue; solo un valor previo lo prueba (paridad) |
| Curvatura de cara y platina en las bandas | — | Pendiente |
| $F$ en el centro de la cara con su incertidumbre | Bloque medido en el centro ($x_c \pm 75$ px); solo la recta de la platina se prolonga hasta $x_c$; $u_\text{pos}(F)$ por la posición y $u_\text{ext}(F)$ = 0,065 franjas por la prolongación (3.11) | **Hecho** (2026-10-29); la cota $u_\text{ext}$ sale de una sola comprobación, y sustituirla exige ver platina a ambos lados del bloque |
| Dimensiones de las caras | Medirlas en los bloques del INM | **Pendiente**: hoy son 9 × 30 y 9 × 35 mm supuestos |
| Borde con $F$ ≈ 0,4–0,5 sin línea oscura | — | **Abierto**: en lo simulado se parte en dos líneas a ±16 px |
| Borde lejano fuera del bloque (haz que recorta el bloque, línea espuria) | Comparar el ancho con el tamaño de píxel declarado (3.10) | **Protegido** (2026-10-29): en 3 tomas de posición se habría desplazado el centro hasta 100 px en x y 325 px en y |
| Extremos con franjas anchas | — | **Abierto**: el 2 mm con 10 franjas da el extremo 50 px arriba, sin aviso |

**Propuestas que había para cerrar los fallos silenciosos** (2026-10-27):

1. **Usar la línea oscura.** Es la que se aplicó el 2026-10-28, como confirmación (3.6) y como respaldo cuando ninguna
   línea de fase la lleva (3.7).
2. **Subir el mínimo de franjas** de la demodulación del campo completo. Ya no hace falta para el borde.
3. **Comparar con la heurística** cuando la geometría del banco está declarada. Tampoco hace falta: el balance de la
   sección 5 la usa como referencia, y el método general coincide con ella.